# Tigers & Goats: Control Center
**Version 1.2** | **Last updated: 2026-10-28** | `TnG_Notebook.ipynb`

One notebook to **inspect, train, evaluate, diagnose and play** Tigers & Goats. It merges the v1.0
training notebook with the older control hub (`x96_TnG.ipynb`). It adds **plain PPO with a configurable
loss** alongside Maskable PPO, and covers the options of the team's v1.1 notebooks (`TnG_MPPO_Notebook.ipynb`,
`TnG_PPO_Notebook.ipynb`):
- several training seeds per run
- masked vs unmasked runs side by side
- an illegal move that either ends the game or costs a turn
- ReLU or tanh activation
- evaluation with and without the mask

The notebook is standalone: it defines the environment itself and runs in Colab or Jupyter without a
clone. Inside the repository it also finds the shipped models in `stable_models/` and can launch `gui.py`.

### How to use it

1. **[Choose what to run](#choose-features):** tick the features you want, or pick a preset.
2. **[Options](#options):** every setting, grouped by feature (A–I). The defaults run a quick check.
3. **Run all.** Sections whose feature is off print `⏭ skipped` and do nothing. Setup, the environment
   and the shared helpers always run, because the other sections depend on them.

### Sections

| § | Section | Feature switch | Origin |
| --- | --- | --- | --- |
| 1 | [Choose what to run](#choose-features) | — | new |
| 2 | [Options](#options) | — | v1.0 configuration cell, extended |
| 3 | [Setup](#setup) | always | v1.0 + hub |
| 4 | [Environment](#environment) | `ENV_DEMO` for the demo | v1.0 env, hub demo + knobs |
| 5 | [Shared helpers](#helpers) | always | v1.0 helpers, hub model tools, **new PPO losses** |
| 6 | [Models](#models) | `MODELS`, `MODEL_SUMMARY` | hub 3a/3b, `model_summary.py` |
| 7 | [TensorBoard](#tensorboard) | `TENSORBOARD` | v1.0 + historical logs |
| 8 | [Train](#train) | `TRAIN` | v1.0 + plain PPO + model tigers |
| 9 | [Evaluate](#evaluate) | `EVALUATE` | v1.0 evaluation + hub models × opponents matrix |
| 10 | [Watch and diagnose](#watch) | `WATCH_GAME`, `DIAGNOSTICS` | hub 5a and 6, reward breakdown |
| 11 | [Play](#play) | `PLAY_BOARD`, `LAUNCH_GUI` | v1.0 clickable board, hub GUI launcher |
| 12 | [Outputs and next steps](#outputs) | — | v1.0 |

Observations are 25 numbers (23 board cells, goats eaten, phase). There are 115 actions,
`pos * 5 + dir`. Maskable PPO only ever samples legal actions. Plain PPO samples from all 115, so
it has to *learn* which moves are legal. That is what its legality loss is for (§5b).

See [Future Plans](Important_Docs/FUTURE_PLANS.ipynb) for opponent pools, sticky sampling and randomized
starts. These are not enabled here.

<a id="choose-features"></a>

## 1 · Choose what to run

Tick features in the next cell; it renders as checkboxes in Colab. Then **Run all**.

* `PRESET` overrides the ticks when it is not `"custom"`:

  | Preset | Turns on |
  | --- | --- |
  | `"quick_look"` | environment demo, models, watch one game, evaluate the selected model |
  | `"train_and_evaluate"` | TensorBoard, train, evaluate the trained models |
  | `"benchmark"` | models, evaluate, diagnostics (set `EVAL_MODELS = "all"` or a list for a models × opponents table) |
  | `"everything"` | every feature (the GUI only opens on a machine with a display) |

* Every section is independent. `MODEL_SUMMARY`, `WATCH_GAME` and `DIAGNOSTICS` use `SELECTED_MODEL`
  (option group E) whether or not `MODELS` is on. `MODELS` only lists what is available.
* To run a single section by hand, run §1–§5 once, then that section's cells.

In [ ]:
#@title Choose what to run { display-mode: "form" }
PRESET = "custom"        #@param ["custom", "quick_look", "train_and_evaluate", "benchmark", "everything"]

ENV_DEMO      = True     #@param {type:"boolean"}
MODELS        = True     #@param {type:"boolean"}
MODEL_SUMMARY = False    #@param {type:"boolean"}
TENSORBOARD   = False    #@param {type:"boolean"}
TRAIN         = True     #@param {type:"boolean"}
EVALUATE      = True     #@param {type:"boolean"}
WATCH_GAME    = False    #@param {type:"boolean"}
DIAGNOSTICS   = False    #@param {type:"boolean"}
PLAY_BOARD    = False    #@param {type:"boolean"}
LAUNCH_GUI    = False    #@param {type:"boolean"}

FEATURE_NAMES = ["ENV_DEMO", "MODELS", "MODEL_SUMMARY", "TENSORBOARD", "TRAIN", "EVALUATE",
                 "WATCH_GAME", "DIAGNOSTICS", "PLAY_BOARD", "LAUNCH_GUI"]
PRESETS = {
    "quick_look":         {"ENV_DEMO", "MODELS", "WATCH_GAME", "EVALUATE"},
    "train_and_evaluate": {"TENSORBOARD", "TRAIN", "EVALUATE"},
    "benchmark":          {"MODELS", "EVALUATE", "DIAGNOSTICS"},
    "everything":         set(FEATURE_NAMES),
}
if PRESET != "custom" and PRESET not in PRESETS:
    raise ValueError(f"PRESET must be 'custom' or one of {sorted(PRESETS)}.")
FEATURES = ({name: name in PRESETS[PRESET] for name in FEATURE_NAMES} if PRESET != "custom"
            else {name: bool(globals()[name]) for name in FEATURE_NAMES})

print(f"preset: {PRESET}")
for name in FEATURE_NAMES:
    print(f"  {'✅' if FEATURES[name] else '⬜'} {name}")

<a id="options"></a>

## 2 · Options

Every setting lives in the next cell, grouped by the feature that uses it. Settings for a switched-off
feature are ignored. Edit the value to the right of `=`, keep text in quotes, and use `None`
(no quotes) to turn an optional setting off. After changing options, **restart the kernel and Run all**.

| Group | Used by | What it controls |
| --- | --- | --- |
| **A · Experiment** | Train | name, learner role, opponent, steps, device, seed(s), output folder |
| **B · Algorithm and loss** | Train | `"maskable_ppo"` or plain `"ppo"`, plain PPO's loss function (§5b), illegal-move handling |
| **C · PPO hyperparameters** | Train | rollout size, batch, epochs, learning rate, γ, λ, clip, coefficients, network, activation |
| **D · Opponents and curriculum** | Train, Evaluate | opponent models, mixing, reward overrides, `TRAINING_RUNS`, resume |
| **E · Models** | Models, Watch, Diagnostics | where to look for `.zip` models and which one to use |
| **F · Evaluation** | Evaluate | which models, which opponents, games, seeds, masked or raw play |
| **G · Demo, watch, diagnostics** | Env demo, Watch, Diagnostics | opponents, seeds, number of games |
| **H · Play** | Play board, GUI | your side, the opponent, the turn limit |
| **I · Setup** | Setup | package install, where the environment code comes from, repo clone |

**Opponents.** Goat learners face `"greedy"`, `"smart"` or `"model"` Tigers (`TIGER_MODEL_PATH`).
Tiger learners face `"random"` or `"model"` Goats (`GOAT_MODEL_PATH`). Evaluation also accepts a
`.zip` path as an opponent.

**Training runs.** `TRAINING_RUNS = None` trains one baseline. A dictionary trains one model per entry.
A list of phase dictionaries continues one model through a curriculum. Reward overrides merge in this
order: environment defaults → `REWARD_OVERRIDES` → run → phase. Phases do not inherit from the previous
phase. See `Important_Docs/TRAINING_RUN_CONFIGS.md`. A run can set `"algorithm"` beside its phases, so
`{"masked": {"algorithm": "maskable_ppo"}, "plain": {"algorithm": "ppo"}}` trains both on identical settings.
`TRAIN_SEEDS = [42, 43, 44]` trains every run once per seed (folders `<run>_seed_<seed>`).

**Resume.** `RESUME_MODEL_PATH` continues a saved model, keeping its architecture and PPO
hyperparameters. Its algorithm must match the run's algorithm, and only one run and seed may be selected.
For plain PPO, the loss options in B are applied to the resumed model.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# A · EXPERIMENT                                                    (Train)
# ════════════════════════════════════════════════════════════════════════════
EXPERIMENT_NAME = "Test_Model"
LEARNER_ROLE = "goat"            # "goat" or "tiger"
OPPONENT = "greedy"              # Goat: "greedy", "smart", "model"; Tiger: "random", "model"
TIMESTEPS = 2048                 # 2,048 steps checks the workflow; increase for substantive training
DEVICE = "cpu"                   # "cpu" or "cuda" (GPU runtime required)
NUM_ENVS = 8                     # Environments stepped sequentially; does not select CPU cores
SEED = 42                        # Integer from 0 through 2**32 - 1
TRAIN_SEEDS = None               # e.g. [42, 43, 44]: one independent model per seed for every run; None uses SEED
OUTPUT_ROOT = "artifacts"        # Local folder or mounted persistent-storage path
CHECKPOINTS_PER_RUN = 10         # Per phase; 0 disables periodic checkpoints
LOG_INTERVAL = 1                 # Print PPO metrics every this many completed rollouts
PRINT_EVERY_SECONDS = 10         # Print progress while stepping, even before a rollout finishes

# ════════════════════════════════════════════════════════════════════════════
# B · ALGORITHM AND LOSS                                            (Train)
# ════════════════════════════════════════════════════════════════════════════
ALGORITHM = "maskable_ppo"       # "maskable_ppo": samples only legal actions (sb3-contrib MaskablePPO)
                                 # "ppo": plain PPO over all 115 actions, with the loss chosen below
# The settings below apply only when ALGORITHM = "ppo". See §5b for the formulas.
PPO_POLICY_LOSS = "clip"         # "clip": PPO clipped surrogate (standard)
                                 # "kl_penalty": surrogate minus an adaptive KL penalty (PPO-Penalty)
                                 # "spo": Simple Policy Optimization, a quadratic ratio penalty instead of a clip
PPO_VALUE_LOSS = "mse"           # "mse" (standard) or "huber" (less sensitive to large return errors)
PPO_HUBER_DELTA = 1.0            # Huber threshold, used when PPO_VALUE_LOSS = "huber"
PPO_KL_TARGET = 0.01             # Target KL per update for "kl_penalty"; beta doubles/halves around it
PPO_KL_BETA = 1.0                # Starting KL penalty coefficient for "kl_penalty"
PPO_LEGALITY_COEF = 0.5          # Weight of the auxiliary legality loss; 0 turns it off
PPO_LEGALITY_LOSS = "nll"        # "nll": -log P(legal action), "mass": P(illegal action)
ILLEGAL_MOVE_HANDLING = "end_game"  # Plain PPO training: "end_game" (an illegal move ends the game) or
                                    # "lose_turn" (it costs a turn; the board and opponent stay put)
PPO_ILLEGAL_MOVE_REWARD = -3.2   # Reward for an illegal move; None keeps the environment's
                                 # REWARD_INVALID_SOFT/HARD (soft is only -0.05). Try about -1 with "lose_turn".

# ════════════════════════════════════════════════════════════════════════════
# C · PPO HYPERPARAMETERS                                (Train, both algorithms)
# ════════════════════════════════════════════════════════════════════════════
N_STEPS = 128                    # Rollout steps per environment (positive integer)
BATCH_SIZE = 64                  # >1; must divide N_STEPS * NUM_ENVS for fresh models
N_EPOCHS = 5                     # PPO optimization epochs per rollout
LEARNING_RATE = 3e-4             # Positive learning rate
GAMMA = 0.99                     # Discount factor, 0 through 1
GAE_LAMBDA = 0.95                # GAE smoothing, 0 through 1
CLIP_RANGE = 0.2                 # Ratio clip ("clip") or trust-region width ("spo")
ENT_COEF = 0.02                  # Nonnegative entropy coefficient
VF_COEF = 0.5                    # Value-loss weight
MAX_GRAD_NORM = 0.5              # Gradient clipping
NET_ARCH = [256, 256, 256]       # Hidden-layer widths shared by policy and value heads
ACTIVATION = "relu"              # "relu" (the team's notebooks since v1.1) or "tanh" (SB3 default; the shipped models)

# ════════════════════════════════════════════════════════════════════════════
# D · OPPONENTS, MIXING AND CURRICULUM                       (Train, Evaluate)
# ════════════════════════════════════════════════════════════════════════════
GOAT_MODEL_PATH = None           # Saved goat .zip: tiger learner vs "model" Goats, or nonzero goat mix
TIGER_MODEL_PATH = None          # Saved tiger .zip: goat learner vs "model" Tigers
# e.g. GOAT_MODEL_PATH = "stable_models/models/Goats/mppo_RobustGoat030726.zip"
#      TIGER_MODEL_PATH = "stable_models/models/Tigers/mppo_RobustTiger030726.zip"
MIX_PROB = None                  # None: fixed opponent; otherwise probability 0..1, drawn every game
# Goat learner: P(Smart Tiger), otherwise Greedy Tiger.
# Tiger learner: P(Model Goat), otherwise Random Goat. Set GOAT_MODEL_PATH if > 0.
REWARD_OVERRIDES = {}            # Keys from DEFAULT_KNOBS (§4), e.g. {"REWARD_STEP": -0.002, "MAX_TURNS": 100}
RESUME_MODEL_PATH = None         # None: fresh learner; or "path/to/checkpoint.zip" (same ALGORITHM)

# TRAINING_RUNS = {run_name: run_config}. Each entry trains a separate model; the key names it.
#   None         = use the settings above
#   {...}        = one phase with some settings changed
#   [{...}, ...] = phases in order, continuing the same model (a curriculum)
#   {"timesteps": ..., "phases": [{...}, ...]} = shared settings beside a phase list
# Phase keys: timesteps, opponent, mix_prob, checkpoints_per_run, goat_model_path,
#             tiger_model_path, reward_overrides. No settings carry over from the previous phase.
# Run-level key (beside "phases", never inside one): "algorithm", overriding ALGORITHM for that run.
TRAINING_RUNS = None             # One baseline training run using the settings above
# Masked vs plain PPO on identical settings (add TRAIN_SEEDS for several seeds each):
# TRAINING_RUNS = {"masked": {"algorithm": "maskable_ppo"}, "plain": {"algorithm": "ppo"}}
# TRAINING_RUNS = {
#     "GvsGT": {"opponent": "greedy", "timesteps": 6_000_000},
#     "GvsST": {"opponent": "smart", "timesteps": 6_000_000},
#     "GvsMixT": [
#         {"opponent": "greedy", "timesteps": 1_500_000},
#         {"opponent": "smart", "timesteps": 1_500_000},
#         {"mix_prob": 0.3, "timesteps": 1_500_000},   # 30% Smart / 70% Greedy
#         {"mix_prob": 0.5, "timesteps": 1_500_000},
#     ],
# }

# ════════════════════════════════════════════════════════════════════════════
# E · MODELS                                       (Models, Watch, Diagnostics)
# ════════════════════════════════════════════════════════════════════════════
MODEL_DIRS = ["stable_models/models", OUTPUT_ROOT]  # Searched recursively; missing folders are skipped
SHOW_CHECKPOINTS = False         # Also list periodic checkpoints (checkpoints/*.zip)
SELECTED_MODEL = "mppo_SmartGoat030726"   # A file name, a unique fragment, a list index, or a full path

# ════════════════════════════════════════════════════════════════════════════
# F · EVALUATION                                                 (Evaluate)
# ════════════════════════════════════════════════════════════════════════════
EVAL_MODELS = "auto"             # "auto": models trained in this session, otherwise the selected model
                                 # "trained" | "selected" | a list of names/fragments/paths | "all"
EVAL_SETTINGS_PATH = None        # Optional phase_*_settings.json for a single evaluated model; when None,
                                 # settings saved next to a model are found automatically
EVAL_OPPONENTS = ["greedy", "smart"]  # Opponent names and/or opponent .zip paths, tested separately
# EVAL_OPPONENTS = None                # The model's final training matchup, including its mix
# EVAL_OPPONENTS = ["random", "model"] # Tiger learners (model = GOAT_MODEL_PATH)
EVAL_EPISODES = 100              # Per model/opponent; use more for reliable comparisons
EVAL_SEED = 1042                 # Separate from training seed
EVAL_DETERMINISTIC = True        # True: highest-probability action; False: sample
EVAL_USE_MASK = True             # True: choose among legal actions only (how MaskablePPO plays)
                                 # False: raw policy, which may make illegal moves (tests legality)
                                 # [True, False]: both, as separate rows
EVAL_ILLEGAL_MOVE = "end_game"   # Raw play: an illegal move "end_game" or "lose_turn"
EVAL_LOG_CSV = "evaluation_log.csv"  # Appended under OUTPUT_ROOT for every evaluation; None disables

# ════════════════════════════════════════════════════════════════════════════
# G · DEMO, WATCH AND DIAGNOSTICS              (Env demo, Watch, Diagnostics)
# ════════════════════════════════════════════════════════════════════════════
DEMO_TIGER = "smart"             # Random legal goat game against "greedy" or "smart" Tigers
DEMO_SEED = 3
DEMO_SHOW_EVERY = 0              # Print the board every N moves (0 = first and last board only)
WATCH_OPPONENT = "smart"         # Opponent for the selected model; follows the model's role
WATCH_SEED = 7
WATCH_USE_MASK = True            # False: let the raw policy play, illegal moves included
WATCH_SHOW_EVERY = 10            # Print the board every N moves (0 = first and last board only)
DIAG_OPPONENT = "greedy"         # Opponent for the diagnostics games
DIAG_GAMES = 10
DIAG_SEED = 42

# ════════════════════════════════════════════════════════════════════════════
# H · PLAY                                                  (Play board, GUI)
# ════════════════════════════════════════════════════════════════════════════
PLAY_AS = "goat"                 # "goat" or "tiger"
PLAY_OPPONENT = "smart"          # You as goat: "greedy", "smart", "model" (TIGER_MODEL_PATH)
                                 # You as tiger: "random", "model" (GOAT_MODEL_PATH)
PLAY_MAX_TURNS = 100             # Positive turn limit for the human game
GUI_TIGER = "smart"              # gui.py scripted tiger: "greedy" or "smart"
GUI_USE_SELECTED_MODEL = True    # Load the selected goat model in the GUI for model moves

# ════════════════════════════════════════════════════════════════════════════
# I · SETUP                                                        (Setup)
# ════════════════════════════════════════════════════════════════════════════
INSTALL_PACKAGES = "auto"        # "auto": install only what is missing; "always"; "never"
ENV_SOURCE = "notebook"          # "notebook": the environment defined in §4
                                 # "repo": import env_tng.py from the repository instead
REPO_DIR = None                  # None: search the working folder and its parents for env_tng.py
CLONE_REPO = False               # True: shallow-clone the repository when it is not found (e.g. Colab)
REPO_URL = "https://github.com/cfalcon-808/TNG_Falcon.git"
REPO_BRANCH = "TnG---Organized"
TB_INCLUDE_HISTORICAL = True     # TensorBoard also shows stable_models/Logging when it exists

<a id="setup"></a>

## 3 · Setup (always runs)

This cell does four things:

* **Packages:** installs any that are missing (`INSTALL_PACKAGES`). If installation asks for a
  restart, restart the kernel and Run all again.
* **Imports:** imports everything the notebook uses.
* **Repository:** finds a repository checkout for the shipped models and `gui.py`. It can
  shallow-clone one if `CLONE_REPO = True`.
* **Section switches:** registers the `%%feature` cell magic. Each section cell starts with
  `%%feature NAME` and runs only when that feature is on.

Run it after every kernel restart. It works the same in Colab, Jupyter and VS Code.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from IPython import get_ipython

# Shared notebook environment check; reused by TensorBoard, human play and the GUI launcher.
shell = get_ipython()
in_colab = "google.colab" in sys.modules or type(shell).__module__.startswith("google.colab")

# ── Packages ────────────────────────────────────────────────────────────────
PACKAGES = {  # import name -> pip requirement
    "stable_baselines3": "stable-baselines3[extra]>=2.7,<3",
    "sb3_contrib": "sb3-contrib>=2.7,<3",
    "gymnasium": "gymnasium>=1.2,<2",
    "torch": "torch",
    "numpy": "numpy",
    "pandas": "pandas",
    "tensorboard": "tensorboard",
}
if not in_colab:
    PACKAGES["ipywidgets"] = "ipywidgets==8.1.8"  # 8.1.9 shows blank widgets in VS Code

if INSTALL_PACKAGES not in {"auto", "always", "never"}:
    raise ValueError('INSTALL_PACKAGES must be "auto", "always" or "never".')
_missing = [req for mod, req in PACKAGES.items() if importlib.util.find_spec(mod) is None]
_to_install = list(PACKAGES.values()) if INSTALL_PACKAGES == "always" else _missing
if _to_install and INSTALL_PACKAGES != "never":
    print("Installing:", ", ".join(_to_install))
    shell.run_line_magic("pip", "install " + " ".join(f'"{req}"' for req in _to_install))
elif _missing:
    raise ImportError(f"Missing packages and INSTALL_PACKAGES = 'never': {_missing}")

# ── Imports shared by every section ─────────────────────────────────────────
import copy
import csv
import json
import math
import random
import re
import shlex
import zipfile
from collections import Counter, deque
from datetime import datetime, timezone
from html import escape
from time import monotonic, perf_counter
from urllib.request import urlopen
from uuid import uuid4

import gymnasium as gym
import ipywidgets as widgets
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from gymnasium import spaces
from IPython.core.magic import register_cell_magic
from IPython.display import IFrame, display
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.buffers import MaskableRolloutBuffer
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.logger import configure
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import explained_variance, set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv

if in_colab:
    from google.colab import output

# ── Repository checkout (optional) ──────────────────────────────────────────
def find_repo():
    if REPO_DIR is not None:
        path = Path(REPO_DIR).expanduser().resolve()
        if not (path / "env_tng.py").is_file():
            raise ValueError(f"REPO_DIR has no env_tng.py: {path}")
        return path
    for folder in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "TNG_Falcon"]:
        if (folder / "env_tng.py").is_file():
            return folder.resolve()
    return None


REPO = find_repo()
if REPO is None and CLONE_REPO:
    target = Path.cwd() / "TNG_Falcon"
    print(f"Cloning {REPO_URL} ({REPO_BRANCH}) into {target} ...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(target)], check=True)
    REPO = find_repo()


def resolve_path(value):
    """A path as given, or relative to the repository when it only exists there."""
    path = Path(value).expanduser()
    if not path.is_absolute() and not path.exists() and REPO is not None and (REPO / path).exists():
        path = REPO / path
    return path


# ── Section switches ────────────────────────────────────────────────────────
@register_cell_magic
def feature(line, cell):
    """%%feature NAME [NAME ...]: run this cell only when every named feature is on (§1)."""
    names = line.split()
    unknown = [name for name in names if name not in FEATURES]
    if not names or unknown:
        raise ValueError(f"%%feature needs names from {FEATURE_NAMES}; got {line!r}")
    off = [name for name in names if not FEATURES[name]]
    if off:
        print(f"⏭ skipped: {' and '.join(off)} {'is' if len(off) == 1 else 'are'} off (§1)")
        return
    # Run the body like a normal cell (magics allowed) so errors still stop "Run all".
    exec(compile(shell.transform_cell(cell), f"<%%feature {line.strip()}>", "exec"), shell.user_ns)


import stable_baselines3, sb3_contrib
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | stable-baselines3 {stable_baselines3.__version__} "
      f"| sb3-contrib {sb3_contrib.__version__} | {'Colab' if in_colab else 'local'}")
print(f"repository: {REPO or 'not found (shipped models and gui.py unavailable; set CLONE_REPO = True to fetch them)'}")

<a id="environment"></a>

## 4 · Environment (always runs; demo with `ENV_DEMO`)

One **Gymnasium environment, `TnGEnv`**, serves both learner roles: goat placement and movement,
tiger captures, legal-action masks, scripted and model opponents, and configurable rewards.
The code cell below defines it (or `ENV_SOURCE = "repo"` imports `env_tng.py` instead).

### Board Map

The board has **23 playable positions**, indexed `0–22` in the `board` array.
Three tigers start at positions `0`, `3`, and `4`. Goats place **15 pieces** before
entering the movement phase. Tigers win by capturing **6 goats**.

| | | 0 (b0) | | | |
| --- | --- | --- | --- | --- | --- |
| 1 (a1) | 2 (b1) | 3 (c1) | 4 (d1) | 5 (e1) | 6 (f1) |
| 7 (a2) | 8 (b2) | 9 (c2) | 10 (d2) | 11 (e2) | 12 (f2) |
| 13 (a3) | 14 (b3) | 15 (c3) | 16 (d3) | 17 (e3) | 18 (f3) |
| | 19 (b4) | 20 (c4) | 21 (d4) | 22 (e4) | |

`render()` prints an ASCII board: `.` = empty, `G` = goat, `T` = tiger.
The environment's `move_map` defines connections and directions; the table is a
position reference, not a full drawing of the legal edges.

### Observations and Actions

- **Observation:** a 25-element vector, `[board(23), goats_eaten, phase]`.
  Board values are `0=empty`, `1=goat`, `2=tiger`; phase is `0=placement`, `1=movement`.
- **Action space:** `Discrete(115)`, encoded as `pos * 5 + dir_code`.
  Decode with `pos = action // 5` and `dir_code = action % 5`.
- **Goat actions:** direction `0` places a goat during placement. Directions `1–4`
  identify movement through `move_map` during the movement phase.
- **Tiger actions:** select a tiger's position and legal direction. The environment
  resolves whether that direction is an ordinary move or a capture, including hub rules.
- **Masks:** `get_action_mask()` returns 115 Boolean values for the current learner.
  The notebook wraps this with `ActionMasker`. Maskable PPO samples only legal
  actions; plain PPO stores the mask to train its legality loss (§5b).

### Environment Functions

#### `TnGEnv(...)`

Important constructor arguments are `learner_role`, `tiger_ai`, `goat_opponent_ai`,
`reward_weights`, and optional `max_turns`. The constructor defaults to a goat learner
against Greedy Tigers; **this notebook's configuration also defaults to Greedy Tigers**.
The training helpers translate the configuration into these arguments.

`max_turns=None` uses `DEFAULT_KNOBS["MAX_TURNS"]`, currently **100**. Override it
through `REWARD_OVERRIDES` in the configuration cell. Model opponents use prediction
callbacks supplied by the helpers when a model file is configured.

#### `reset(seed=None, options=None)`

Resets the board, placement phase, counters, and move bookkeeping, and returns
`(observation, info)`. The notebook seeds Python, NumPy, and PyTorch as well as the
environment, because the scripted opponents also use global random generators.

#### `get_state()` and `render()`

`get_state()` returns the observation vector. `render()` prints the board and game
status; no interactive GUI is required.

#### `step(action)`

Returns `(observation, reward, terminated, truncated, info)`.

- **Goat learner:** the agent places or moves a goat, then the scripted tiger responds
  unless a terminal condition has already been reached.
- **Tiger learner:** the agent moves a tiger, then a Random or Model Goat responds
  unless a terminal condition has already been reached.
- Goats win when tigers have no legal moves. Tigers win at the capture threshold;
  the tiger-learner path also awards a tiger win when goats have no legal response.
- The turn limit ends an episode with `winner="MaxTimeout"`. The goat-learner path
  also detects repeated-state stalls, reported as `winner="RepeatTimeout"`.
- Invalid actions have separate soft/hard handling and can truncate an episode.
  Action masking prevents them for Maskable PPO. Plain PPO training treats an illegal
  move as the end of the game (§5b). Inspect `info["reason"]` for details.

### Opponents and Mixing

| Learner | Configuration opponent | Behavior |
| --- | --- | --- |
| Goat | `"greedy"` | Capture-biased random Tiger: with default capture bias 1.0, takes a random available capture; otherwise selects a legal move |
| Goat | `"model"` | Uses a saved Tiger policy (`TIGER_MODEL_PATH`), choosing its most likely legal move |
| Goat | `"smart"` | Heuristic Tiger: prioritizes captures, anchor positions `0` and `9/10`, then stronger board squares |
| Tiger | `"random"` | Chooses a random legal Goat action |
| Tiger | `"model"` | Uses a saved Goat policy (`GOAT_MODEL_PATH`), choosing its most likely legal move |

The Tiger policy methods are `_greedy_tiger_move(...)` and `_smart_tiger_move(...)`.
All five matchups can be used for training, evaluation and play. Saved opponents may be
Maskable PPO or plain PPO models.

`MIX_PROB=None` uses a fixed opponent. Otherwise, at each reset, it selects Smart
instead of Greedy Tigers with that probability for a goat learner, or Model instead
of Random Goats for a tiger learner. A nonzero Model Goat probability requires a
valid model file. The helpers reject missing model files. Within the environment, a failed
or illegal Model Goat prediction falls back to a random legal Goat action. A failed or illegal
Model Tiger prediction raises an error (no fallback). The notebook's model opponents always
choose among legal moves, so this does not happen in practice.

### Rewards

Although the environment's reward entry point is named `sparse_reward`, the current
implementation includes **role-specific shaping**, not just terminal rewards.

- **Goat learner:** step costs, tiger mobility, territory, tiger spread, center control,
  capture penalties, repetition penalties, and terminal outcomes. Some positive
  shaping decays during the movement phase.
- **Tiger learner:** step costs, capture rewards, mobility and center shaping, plus
  terminal win/loss and timeout terms.

Selected defaults (these are components, not necessarily the total step reward):

| Setting | Default effect |
| --- | --- |
| `REWARD_GOAT_WIN` | Goat win reward starts at `3.2`, reduced by `GOAT_WIN_TURN_DECAY * turns` (decay default `0.01`) |
| `REWARD_TIGER_WIN` | Goat learner's terminal loss component: `-3.2` |
| `REWARD_GOAT_EATEN` | Goat learner: `-0.35` per captured goat |
| `REWARD_TIGER_CAPTURE` | Tiger learner: `+0.35` per capture |
| `REWARD_TIGER_WIN_BONUS` | Tiger learner: `+3.2` terminal win bonus |
| `REWARD_TIGER_LOSS_PENALTY` | Tiger learner: subtract `3.2` on a goat win |
| `MAX_TIMEOUT_SCALE` | Turn-limit penalty scales the role's terminal loss term by `1.5` |
| `REPEAT_STALL_SCALE` | Goat learner's repeated-state timeout scales its loss term by `1.25` |

Use `REWARD_OVERRIDES` for custom rewards and game limits. The **Environment and
reward reference** section below lists every supported key, default, and meaning.

### Training and Evaluation Workflow

The default run trains a goat against Greedy Tigers for **2,048 timesteps** on CPU.
Training uses Maskable PPO or plain PPO with a configurable loss (§5b), sequential vectorized environments, TensorBoard logging,
periodic checkpoints, and final saved models. Training runs start independently;
curriculum phases continue the same model. Saved checkpoints can resume training.

The evaluation section runs separately, defaults to **100 episodes**, and can
evaluate newly trained or existing models. It reports wins, losses, timeouts, mean
reward, mean episode length and illegal moves, and exports CSV/JSON results. A short training run
checks the workflow; use the configuration cell to increase training and evaluation
budgets before assessing policy strength.

### 4a · Reward and game settings reference

The code cell below defines `TnGEnv`. Running it does not start training. Change these defaults
with `REWARD_OVERRIDES` (options group D); §4b shows the values in effect.

### Reward and game settings

| Key | Default | Meaning |
| --- | --- | --- |
| `REWARD_GOAT_WIN` | `3.2` | reward when goats immobilize all tigers |
| `REWARD_TIGER_WIN` | `-3.2` | penalty when tigers win by eating goats |
| `REWARD_TIGER_CAPTURE` | `0.35` | reward per goat captured (tiger learner) |
| `REWARD_TIGER_WIN_BONUS` | `3.2` | reward for tiger win (tiger learner) |
| `REWARD_TIGER_LOSS_PENALTY` | `3.2` | penalty when tiger loses or times out (tiger learner) |
| `REWARD_STEP` | `-0.001` | per-step penalty to discourage stalling |
| `REWARD_GOAT_EATEN` | `-0.35` | penalty for each goat captured by tigers |
| `MAX_TIMEOUT_SCALE` | `1.5` | scale for turn-limit (max turns) timeout penalty |
| `REPEAT_STALL_SCALE` | `1.25` | scale for stall-timeout (repeat-state) penalty |
| `MAX_TURNS` | `100` | max turns before enforcing max-timeout |
| `MOVE_STEP_BASE` | `-0.01` | base negative reward for moving-phase steps |
| `MOVE_STEP_SLOPE` | `-0.01` | step penalty increases as move_steps grows |
| `MOVE_STEP_MIN` | `-0.6` | minimum move penalty (lower bound) |
| `GOAT_WIN_TURN_DECAY` | `0.01` | goat win reward reduced slightly per turn |
| `REWARD_BLOCK_TIGER` | `0.08` | reward for reducing tiger mobility |
| `REWARD_BUBBLE_SPACE` | `0.02` | reward for increasing goat "bubble" territory |
| `REWARD_CLUSTER_TIGERS` | `0.02` | reward for spreading tigers apart |
| `REWARD_CENTER_GOAT` | `0.03` | reward for goats occupying strong center nodes |
| `REWARD_CENTER_TIGER` | `-0.03` | penalty for tigers holding center nodes |
| `NEAR_LOCK_BONUS` | `0.05` | bonus when goats push tigers into near-lock states |
| `LATE_GAME_START_TURN` | `40` | turn at which late-game shaping begins to scale |
| `MOBILITY_BACKSLIDE_SCALE` | `0.5` | penalty multiplier when goats worsen tiger mobility |
| `GOAT_SHAPING_DECAY_BASE` | `0.95` | exponential decay for positive goat shaping in moving phase |
| `REWARD_INVALID_SOFT` | `-0.05` | small penalty for soft invalid actions |
| `REWARD_INVALID_HARD` | `-1.0` | large penalty for hard invalid actions (terminate) |
| `REWARD_REPEAT_STATE` | `-0.5` | (Quadratic along count) penalty for repeating a previous board state |
| `MAX_REPEATS` | `4` | how many repeats before a stall timeout triggers |
| `BASE_TIGER_CAPTURE_BIAS` | `1.0` | how greedy tigers are toward captures (1.0 = full greedy) |

### Environment interface and extension points

The environment provides one `Discrete(115)` interface for both learner roles,
legal-action masks, placement/movement rules, repetition handling, and time limits.
`reset()` returns `(observation, info)`; `step()` returns
`(observation, reward, terminated, truncated, info)`. Game wins are terminal outcomes;
limits and invalid-action endings use truncation as described in the overview.
The `info` dictionary carries outcome/reason, role, move, and reward-component data
where supplied by the relevant transition and reward function.

For advanced environment customization, the constructor accepts `reward_fn` and
stores it as `self.reward_fn`. The default is the role-aware `sparse_reward` method,
which includes the shaping described above. A custom reward callable receives
`(previous_observation, action, observation, terminated, truncated, info)` and returns
one scalar reward. This requires editing environment construction in the helpers;
ordinary experiments should use `REWARD_OVERRIDES` in the configuration cell.

The environment also supports `tiger_ai="model"` with a `tiger_model_predict_fn` callback.
The notebook exposes it as `OPPONENT = "model"` with `TIGER_MODEL_PATH`.

**v1.1 environment changes.** A tiger learner's illegal move used to *earn*
`+abs(REWARD_INVALID_HARD)` (+1.0); it now costs `-abs(REWARD_INVALID_HARD)`. Masked training never
reached this path, but unmasked training would have learned to exploit it. The new constructor
argument `truncate_on_invalid=False` makes an illegal move cost a turn instead of ending the game
(`ILLEGAL_MOVE_HANDLING = "lose_turn"`).

### Environment construction examples

These examples explain the API; they are not extra configuration cells. They become
available after executing the environment definition below:

```python
env = TnGEnv()  # Goat learner against Greedy Tigers
env = TnGEnv(tiger_ai=TIGER_AI_SMART)  # Goat learner against Smart Tigers
env = TnGEnv(learner_role=TIGER_LEARNER)  # Tiger learner against Random Goats
env = TnGEnv(max_turns=150)  # Explicit turn limit
env = TnGEnv(reward_weights={"REWARD_BUBBLE_SPACE": 0.0})  # Disable this shaping term
```

Choose one construction, call `env.reset()`, use the legal-action mask for steps,
and call `env.close()` when finished. The notebook's training/evaluation helpers
perform that lifecycle for normal experiments.

In [ ]:
#  GLOBAL CONSTANTS - Game Rules

BOARD_SIZE                = 23          # board positions (0-22)
DIR_CODES                 = 5           # direction codes (0-4)
TOTAL_GOATS_TO_PLACE      = 15          # goats placed before moving phase begins
GOATS_EATEN_FOR_TIGER_WIN = 6           # tiger wins after this many goats eaten
TIGER_START_POSITIONS     = [0, 3, 4]   # starting indices for tigers
KEY_CENTERS               = [0, 9, 10, 15, 16] # top hub + center spots to control

# Debug flag to avoid slow prints in training
DEBUG_INVALID = False

TIGER_AI_GREEDY = "greedy"
TIGER_AI_SMART = "smart"
TIGER_AI_MODEL = "model"
VALID_TIGER_AI = {TIGER_AI_GREEDY, TIGER_AI_SMART, TIGER_AI_MODEL}
GOAT_LEARNER = "goat"
TIGER_LEARNER = "tiger"
GOAT_AI_RANDOM = "random"
GOAT_AI_MODEL = "model"


#  GLOBAL TUNING KNOBS
#  Configurable reward magnitudes, shaping scales, and episode limits.
#  Override these values with REWARD_OVERRIDES in the configuration code cell.

DEFAULT_KNOBS = {

    # Goat-learner terminal rewards
    "REWARD_GOAT_WIN":          3.2,    # reward when goats immobilize all tigers
    "REWARD_TIGER_WIN":        -3.2,    # penalty when tigers win by eating goats

    # Tiger-learner rewards (positive magnitudes)
    "REWARD_TIGER_CAPTURE":     0.35,   # reward per goat captured (tiger learner)
    "REWARD_TIGER_WIN_BONUS":   3.2,    # reward for tiger win (tiger learner)
    "REWARD_TIGER_LOSS_PENALTY": 3.2,   # penalty when tiger loses or times out (tiger learner)

    # Goat-learner step & capture shaping
    "REWARD_STEP":             -0.001,  # per-step penalty to discourage stalling
    "REWARD_GOAT_EATEN":       -0.35,   # penalty for each goat captured by tigers

    # Timeout scales
    "MAX_TIMEOUT_SCALE":        1.5,    # scale for turn-limit (max turns) timeout penalty
    "REPEAT_STALL_SCALE":       1.25,   # scale for stall-timeout (repeat-state) penalty
    "MAX_TURNS":                100,    # max turns before enforcing max-timeout

    # Move penalty shaping
    "MOVE_STEP_BASE":          -0.01,   # base negative reward for moving-phase steps
    "MOVE_STEP_SLOPE":         -0.01,   # step penalty increases as move_steps grows
    "MOVE_STEP_MIN":           -0.6,    # minimum move penalty (lower bound)

    # Decay for goat win  
    "GOAT_WIN_TURN_DECAY":      0.01,   # goat win reward reduced slightly per turn

    # Shaping magnitudes  
    "REWARD_BLOCK_TIGER":       0.08,   # reward for reducing tiger mobility
    "REWARD_BUBBLE_SPACE":      0.02,   # reward for increasing goat "bubble" territory
    "REWARD_CLUSTER_TIGERS":    0.02,   # reward for spreading tigers apart
    "REWARD_CENTER_GOAT":       0.03,   # reward for goats occupying strong center nodes
    "REWARD_CENTER_TIGER":     -0.03,   # penalty for tigers holding center nodes

    # Special shaping
    "NEAR_LOCK_BONUS":          0.05,   # bonus when goats push tigers into near-lock states
    "LATE_GAME_START_TURN":     40,     # turn at which late-game shaping begins to scale
    "MOBILITY_BACKSLIDE_SCALE": 0.5,    # penalty multiplier when goats worsen tiger mobility
    "GOAT_SHAPING_DECAY_BASE":  0.95,   # exponential decay for positive goat shaping in moving phase

    # Anti-repeat penalty
    "REWARD_INVALID_SOFT":     -0.05,   # small penalty for soft invalid actions
    "REWARD_INVALID_HARD":     -1.0,    # large penalty for hard invalid actions (terminate)
    "REWARD_REPEAT_STATE":     -0.5,    # (Quadratic along count) penalty for repeating a previous board state
    "MAX_REPEATS":              4,      # how many repeats before a stall timeout triggers

    # Tiger AI difficulty
    "BASE_TIGER_CAPTURE_BIAS": 1.0,     # how greedy tigers are toward captures (1.0 = full greedy)
}


#  BOARD COORDINATES - Human-Readable Labels

COORD_LABELS = [
    "b0",                                    # 0
    "a1", "b1", "c1", "d1", "e1", "f1",      # 1-6
    "a2", "b2", "c2", "d2", "e2", "f2",      # 7-12
    "a3", "b3", "c3", "d3", "e3", "f3",      # 13-18
    "b4", "c4", "d4", "e4",                  # 19-22
]

# Special hub capture jumps from node 0 (b0)
HUB0_CAPTURE_JUMP = {
    1: 8,   # 0 -> 2 (b1) -> 8 (b2)
    2: 9,   # 0 -> 3 (c1) -> 9 (c2)
    3: 10,  # 0 -> 4 (d1) -> 10 (d2)
    4: 11,  # 0 -> 5 (e1) -> 11 (e2)
}


#  Board index to coordinate helper

def idx_to_coord(idx: int) -> str:
    """Safely convert a board index to its coordinate label."""
    if 0 <= idx < len(COORD_LABELS):
        return COORD_LABELS[idx]
    return str(idx)


#  BASE ENVIRONMENT

class TnGEnv(gym.Env):
    """
    Tigers and Goats Environment.

    Observation (Box(25,)):
      - 23 board cells: 0=empty, 1=goat, 2=tiger
      - 1 value: goats eaten (0..6)
      - 1 value: phase (0=placing, 1=moving)

    Action (Discrete(115)):
      - flat action index in [0, 114]
      - decode rule: pos = action // DIR_CODES, dir_code = action % DIR_CODES
    """

    #  Group: Core Setup and Configuration
    #  Initializes persistent env state and shared tuning knobs.
    #  These members are prerequisites for all runtime behavior.
    def __init__(
            self, 
            reward_weights=None, 
            tiger_ai: str = TIGER_AI_GREEDY, 
            learner_role: str = GOAT_LEARNER, 
            goat_opponent_ai: str = GOAT_AI_RANDOM, 
            goat_model_predict_fn=None, 
            tiger_model_predict_fn=None,
            reward_fn=None,
            max_turns: int | None = None,
            truncate_on_invalid: bool = True,
            ):
        """Initialize environment state, tuning knobs, spaces, and board topology."""
        super(TnGEnv, self).__init__()

        # Numeric tuning knobs (constants + shaping magnitudes + scales)
        self.knobs = dict(DEFAULT_KNOBS)

        self.learner_role = (learner_role or GOAT_LEARNER).lower()
        self.goat_opponent_ai = (goat_opponent_ai or GOAT_AI_RANDOM).lower()
        self._goat_model_predict_fn = goat_model_predict_fn  # callable(obs, mask)->flat action
        self._tiger_model_predict_fn = tiger_model_predict_fn  # callable(obs, mask)->flat action
        self.reward_fn = reward_fn or self.sparse_reward
        self.truncate_on_invalid = truncate_on_invalid

        # Load any tuning values passed to this function.
        if reward_weights is not None:
            for key, val in reward_weights.items():
                if key in self.knobs:
                    self.knobs[key] = val
                else:
                    print(f"[warning] Unknown tuning key: {key}")

        configured_max_turns = self.knobs["MAX_TURNS"] if max_turns is None else max_turns
        self.max_turns = int(configured_max_turns)
        if self.max_turns <= 0:
            raise ValueError(f"max_turns must be > 0, got: {configured_max_turns}")
        # Keep _w("MAX_TURNS") and explicit max_turns aligned.
        self.knobs["MAX_TURNS"] = self.max_turns

        self.tiger_ai = (tiger_ai or TIGER_AI_GREEDY).lower()
        if self.tiger_ai not in VALID_TIGER_AI:
            raise ValueError(f"tiger_ai must be one of {VALID_TIGER_AI}")
        if self.learner_role not in {GOAT_LEARNER, TIGER_LEARNER}:
            raise ValueError(f"learner_role must be one of: {GOAT_LEARNER}, {TIGER_LEARNER}")
        if self.goat_opponent_ai not in {GOAT_AI_RANDOM, GOAT_AI_MODEL}:
            raise ValueError(f"goat_opponent_ai must be one of: {GOAT_AI_RANDOM}, {GOAT_AI_MODEL}")

        # Observation space
        self.observation_space = spaces.Box(
            low=np.array([0] * BOARD_SIZE + [0] + [0], dtype=np.int8),
            high=np.array([2] * BOARD_SIZE + [6] + [1], dtype=np.int8),
            shape=(BOARD_SIZE + 2,),
            dtype=np.int8,
        )

        # Action space (flat)
        self.action_space = spaces.Discrete(BOARD_SIZE * DIR_CODES)

        # Game state variables
        self.terminate = False
        self.board = np.zeros(BOARD_SIZE, dtype=np.int8)
        for idx in TIGER_START_POSITIONS:
            self.board[idx] = 2

        self.eaten = 0          # goats eaten by tigers
        self.phase = 0          # 0 = placing phase, 1 = moving phase
        self.goats_placed = 0

        # For reward shaping
        self.prev_tiger_moves = 0
        self.prev_unreachable = 0
        self.prev_tiger_spread = 0.0

        # Repetition tracker
        self.turns = 0
        self.move_steps = 0     # Counts moves taken in move phase
        self.state_history = {}

        # For human-readable move tracking in render()
        self.last_move_desc = ""

        # Move map:
        # pos -> {dir_code: destination_pos}
        self.move_map = {
            0: {1: 2, 2: 3, 3: 4, 4: 5},  # b0 special
            1: {2: 2, 3: 7},                 # a1
            2: {1: 0, 2: 3, 3: 8, 4: 1},     # b1
            3: {1: 0, 2: 4, 3: 9, 4: 2},     # c1
            4: {1: 0, 2: 5, 3: 10, 4: 3},    # d1
            5: {1: 0, 2: 6, 3: 11, 4: 4},    # e1
            6: {3: 12, 4: 5},                # f1

            7: {1: 1, 2: 8, 3: 13},          # a2
            8: {1: 2, 2: 9, 3: 14, 4: 7},    # b2
            9: {1: 3, 2: 10, 3: 15, 4: 8},   # c2
            10: {1: 4, 2: 11, 3: 16, 4: 9},  # d2
            11: {1: 5, 2: 12, 3: 17, 4: 10}, # e2
            12: {1: 6, 3: 18, 4: 11},        # f2

            13: {1: 7, 2: 14},               # a3
            14: {1: 8, 2: 15, 3: 19, 4: 13}, # b3
            15: {1: 9, 2: 16, 3: 20, 4: 14}, # c3
            16: {1: 10, 2: 17, 3: 21, 4: 15},# d3
            17: {1: 11, 2: 18, 3: 22, 4: 16},# e3
            18: {1: 12, 4: 17},              # f3

            19: {1: 14, 2: 20},              # b4
            20: {1: 15, 2: 21, 4: 19},       # c4
            21: {1: 16, 2: 22, 4: 20},       # d4
            22: {1: 17, 4: 21},              # e4
        }

        # Precompute all-pairs shortest path lengths on the static board graph
        self._dist_matrix = self._precompute_all_pairs_dist()

        # Pre-allocated visited buffer for bubble BFS
        self._visited_buffer = np.zeros(BOARD_SIZE, dtype=bool)

        # Initial valid moves
        self.valid_moves = []
        self._update_valid_moves()
    # end def __init__()

    def _w(self, key) -> float:
        """
        Knob accessor:
            - If key is a known tuning knob -> return value
            - If unknown -> throw an error
        """
        if key in self.knobs:
            return self.knobs[key]
        
        raise KeyError(
            f"[TnGEnv] Unknown tuning key '{key}'. "
            f"Valid knob keys: {list(self.knobs.keys())}."
        )


    #  Group: Public Gym API
    #  Entry points expected by Gymnasium and training loops.
    #  Keeps reset/step/state/render/mask behavior in one place.
    def reset(self, seed=None, options=None):
        """Reset board, game state, and valid moves."""
        super().reset(seed=seed)

        # Normalize + validate tiger_ai (important if wrappers modify it between episodes)
        self.tiger_ai = (self.tiger_ai or TIGER_AI_GREEDY).lower()
        if self.tiger_ai not in VALID_TIGER_AI:
            raise ValueError(f"tiger_ai must be one of {VALID_TIGER_AI}")

        # Snapshot tiger used for THIS episode (critical for mixed training)
        self._episode_tiger_ai = self.tiger_ai

        self.board[:] = 0
        for idx in TIGER_START_POSITIONS:
            self.board[idx] = 2

        self.eaten = 0
        self.phase = 0
        self.goats_placed = 0
        self.terminate = False

        self.turns = 0
        self.move_steps = 0
        self.state_history = {}

        # clear last move description
        self.last_move_desc = ""

        # initial tiger mobility + geometry metrics
        tiger_opts = self._tiger_moves()
        self.prev_tiger_moves = len(tiger_opts)
        self.prev_unreachable = self._compute_unreachable_safe_cells()
        self.prev_tiger_spread = self._tiger_spread()

        self._update_valid_moves()

        obs = self.get_state()
        info = {
            "action_mask": self.get_action_mask(),
            "tiger_ai": self.tiger_ai,
            "tiger_ai_id": float(1 if self.tiger_ai == TIGER_AI_SMART else 0),
            "episode_tiger_ai": self._episode_tiger_ai,
            "episode_tiger_ai_id": float(1 if self._episode_tiger_ai == TIGER_AI_SMART else 0)
            }
        return obs, info

    def step(self, action):
        """Gym step: execute learner-role transition, then shape reward from step info."""
        prev_obs = self.get_state().copy()
        if self.learner_role == TIGER_LEARNER:
            result = self._step_tiger_transition(action)
        else:
            result = self._step_goat_transition(action)
        if not self.truncate_on_invalid and result[4].get("invalid_action", False):
            # Illegal attempts consume a turn but leave the board and opponent idle.
            obs, reward, terminated, truncated, info = result
            self.terminate = self.turns >= self.max_turns
            if self.terminate:
                info["invalid_reason"] = info["reason"]
                info.update(reason="max_timeout", winner="MaxTimeout")
            result = obs, reward, False, self.terminate, info
        return self._apply_reward_fn(prev_obs, action, result)

    def get_state(self):
        """
        Flattened state: [board(23), eaten(1), phase(1)] as int8.
        """
        return np.concatenate(
            (self.board, [self.eaten, self.phase])
        ).astype(np.int8)

    def render(self, show_coords=True):
        """
        Polished ASCII render for easier human analysis.
            - Empty cells shown as '0' instead of '.'
            - Cleaner alignment and spacing
        - Optional coordinate labels
        - Status bar showing game metrics
        """

        # Updated symbol map: ie. empty = 0,...
        symbols = {0: "0", 1: "G", 2: "T"}    
        cells = [symbols[v] for v in self.board]
        goats_eaten = self.eaten
        phase_str = "Place" if self.phase == 0 else "Move"
    
        # pull board coordinates from global COORD_LABELS
        coord = (
            [f" {COORD_LABELS[i]} " for i in range(BOARD_SIZE)] 
            if show_coords else [""] * BOARD_SIZE
        )
    
        # Build each visual line
        line0 = f"                  {cells[0]}{coord[0]}"
        line1 = (
            f"   {cells[1]}{coord[1]} {cells[2]}{coord[2]} "
            f"{cells[3]}{coord[3]} {cells[4]}{coord[4]} "
            f"{cells[5]}{coord[5]} {cells[6]}{coord[6]}"
        )
        line2 = (
            f"   {cells[7]}{coord[7]} {cells[8]}{coord[8]} "
            f"{cells[9]}{coord[9]} {cells[10]}{coord[10]} "
            f"{cells[11]}{coord[11]} {cells[12]}{coord[12]}"
        )
        line3 = (
            f"   {cells[13]}{coord[13]} {cells[14]}{coord[14]} "
            f"{cells[15]}{coord[15]} {cells[16]}{coord[16]} "
            f"{cells[17]}{coord[17]} {cells[18]}{coord[18]}"
        )
        line4 = (
            f"         {cells[19]}{coord[19]} {cells[20]}{coord[20]} "
            f"{cells[21]}{coord[21]} {cells[22]}{coord[22]}"
        )
    
        # Status panel
        status = (
            f"-----------------------------------------\n"
            f" Phase: {phase_str:<5} | Goats Eaten: {goats_eaten} | Turn: {self.turns}\n"
            f"-----------------------------------------"
        )
    
        # Last-move description
        if getattr(self, "last_move_desc",""):
            status += f"\n Last move: {self.last_move_desc}"

        print(
            "\n"
            "============== TIGERS & GOATS BOARD ==============\n"
            f"{line0}\n{line1}\n{line2}\n{line3}\n{line4}\n{status}\n"
        )
    # end def render()

    def get_action_mask(self):
        """
        Returns a Boolean mask of shape (BOARD_SIZE * DIR_CODES,),
        where each index corresponds to (pos, dir_code) = (i // DIR_CODES, i % DIR_CODES).

        - Goat learner: uses self.valid_moves in transition-safe form:
            - pair: [pos, dir_code] or (pos, dir_code)
            - flat: int action id
        - Tiger learner: uses _tiger_moves(include_dir=True) to mark legal tiger actions.
        """
        mask = np.zeros(BOARD_SIZE * DIR_CODES, dtype=bool)

        if getattr(self, "learner_role", GOAT_LEARNER) == TIGER_LEARNER:
            for from_pos, _to_pos, _cap, dir_code in self._tiger_moves(include_dir=True):
                flat = from_pos * DIR_CODES + dir_code
                if 0 <= flat < mask.size:
                    mask[flat] = True
            return mask

        for mv in self.valid_moves:
            if np.isscalar(mv):
                flat = int(mv)
            else:
                pos, dir_code = mv
                flat = self.encode_action(pos, dir_code)
            if 0 <= flat < mask.size:
                mask[flat] = True

        return mask


    #  Group: Action Encoding and Validation
    #  Converts between flat actions and structured moves.
    #  Centralizes defensive parsing/normalization for transitions.
    def encode_action(self, pos: int, dir_code: int) -> int:
        """Map (board position, direction code) -> flat Discrete action id."""
        return int(pos) * DIR_CODES + int(dir_code)

    def decode_action(self, action):
        """Decode a flat Discrete(115) action id into `(pos, dir_code)`."""
        if isinstance(action, np.ndarray):
            arr = np.asarray(action)
            if arr.size != 1:
                raise ValueError(f"Unrecognized action format: {action} (type {type(action)})")
            action = arr.item()

        if not np.isscalar(action):
            raise ValueError(f"Unrecognized action format: {action} (type {type(action)})")

        a = int(action)
        return a // DIR_CODES, a % DIR_CODES

    def _action_to_flat(self, action):
        """Best-effort conversion from a scalar action into a flat action id."""
        if isinstance(action, np.ndarray):
            arr = np.asarray(action)
            if arr.size != 1:
                return None
            action = arr.item()
        if np.isscalar(action):
            return int(action)
        return None

    def _safe_decode_action(self, action):
        """Decode action into (pos, dir) and return None on malformed input."""
        try:
            pos, dir_code = self.decode_action(action)
            return int(pos), int(dir_code)
        except Exception:
            return None

    def _build_step_info(self, **extra):
        """Build a consistent info payload and apply call-site overrides."""
        info = {
            "reason": None,
            "winner": None,
            "goat_action": None,
            "goat_decoded": None,
            "tiger_move": None,
            "phase": int(self.phase),
            "goats_eaten": int(self.eaten),
            "goats_placed": int(self.goats_placed),
            "turn_counter": int(self.turns),
            "learner_role": self.learner_role,
            "action_mask": self.get_action_mask(),
            "tiger_ai": self.tiger_ai,
            "tiger_ai_id": float(1 if self.tiger_ai == TIGER_AI_SMART else 0),
            "episode_tiger_ai": self._episode_tiger_ai,
            "episode_tiger_ai_id": float(1 if self._episode_tiger_ai == TIGER_AI_SMART else 0),
            "goat_opponent_ai": self.goat_opponent_ai,
            "reward_step_component": 0.0,
            "reward_near_lock_component": 0.0,
            "reward_block_component": 0.0,
            "reward_bubble_component": 0.0,
            "reward_cluster_component": 0.0,
            "reward_center_component": 0.0,
            "reward_goat_eaten_component": 0.0,
            "reward_repeat_component": 0.0,
            "reward_terminal_component": 0.0,
            "reward_decay_mult": 1.0,
            "reward_total": 0.0,
        }
        info.update(extra)
        return info

    def _set_reward_components(self, info, **components):
        """Attach scalar reward breakdown fields to info for logging/debugging."""
        for key, value in components.items():
            info[key] = float(value)


    #  Group: Step Info and Reward Pipeline
    #  Builds standardized info payloads and computes shaped rewards.
    #  Keeps reward flow consistent across goat/tiger learner modes.
    def _apply_reward_fn(self, prev_obs, action, result):
        """Run the configured reward function for the transition result."""
        obs, reward, terminated, truncated, info = result
        final_reward = self.reward_fn(
            prev_obs,
            action,
            obs,
            bool(terminated),
            bool(truncated),
            info,
        )
        return obs, float(final_reward), bool(terminated), bool(truncated), info

    def sparse_reward(self, prev_obs, action, obs, terminated, truncated, info):
        """
        Role-aware sparse reward entry point.

        Purpose:
        - The environment supports two training modes (goat learner, tiger learner).
        - Both modes share transition machinery but use different reward objectives.
        - This function routes to the correct reward builder using the role recorded
          in `info` (preferred) with `self.learner_role` as fallback.
        """
        role = info.get("learner_role", self.learner_role)
        if role == TIGER_LEARNER:
            return self._sparse_reward_tiger(prev_obs, action, obs, terminated, truncated, info)
        return self._sparse_reward_goat(prev_obs, action, obs, terminated, truncated, info)

    def _sparse_reward_goat(self, prev_obs, action, obs, terminated, truncated, info):
        """
        Goat-learner reward from transition summary in `info`.

        High-level composition:
        1) Early-return terminal/invalid cases from `reason`
        2) Per-step shaping and tactical shaping
        3) Material and position shaping
        4) Outcome penalties (tiger win / timeout) or repeat-state penalty

        Important `info` fields consumed here:
        - reason, step_penalty, near_lock, late
        - delta_moves, delta_bubble, delta_spread, center_score
        - goats_eaten_this_turn, repeat_prev_count
        """
        reason = info.get("reason")

        # 1) Immediate terminal/invalid outcomes
        # These bypass incremental shaping to keep terminal reward semantics explicit.
        if reason == "already_terminated":
            self._set_reward_components(info, reward_total=0.0)
            return 0.0
        if reason == "invalid_soft":
            reward_total = self._w("REWARD_INVALID_SOFT")
            self._set_reward_components(info, reward_total=reward_total)
            return reward_total
        if reason == "invalid_hard":
            reward_total = self._w("REWARD_INVALID_HARD")
            self._set_reward_components(info, reward_total=reward_total)
            return reward_total
        if reason == "goat_win_no_tiger_moves":
            # Goat terminal win bonus, with optional late-turn decay.
            reward_total = (
                self._w("REWARD_GOAT_WIN")
                - self._w("GOAT_WIN_TURN_DECAY") * self.turns
            )
            self._set_reward_components(
                info,
                reward_terminal_component=reward_total,
                reward_total=reward_total,
            )
            return reward_total

        # 2) Base step + tactical shaping
        # Step penalty comes from step_transition code (placing/moving can differ).
        step_component = float(info.get("step_penalty", self._w("REWARD_STEP")))
        near_lock_component = 0.0
        block_component = 0.0
        bubble_component = 0.0
        cluster_component = 0.0
        center_component = 0.0
        goat_eaten_component = 0.0
        repeat_component = 0.0
        terminal_component = 0.0

        phase = int(info.get("phase", self.phase))
        move_steps = int(info.get("move_steps", self.move_steps))

        # Exponential decay based on number of steps
        decay_mult = (
            self._w("GOAT_SHAPING_DECAY_BASE") ** move_steps
            if phase == 1 else 1.0
        )

        # Get delta and goat eaten stats which are computed in the goat_step_transition function
        
        # Mobility delta is defined from goat perspective:
        #   delta_moves > 0 means goats reduced tiger options (good for goats).
        #   delta_moves < 0 means goats increased tiger options (bad backslide).
        delta_moves = int(info.get("delta_moves", 0))
        delta_bubble = int(info.get("delta_bubble", 0))
        delta_spread = float(info.get("delta_spread", 0.0))
        goats_eaten_this_turn = int(info.get("goats_eaten_this_turn", 0))

        terminal_loss_reason = reason in {
            "tiger_win_capture_threshold",
            "max_timeout",
            "repeat_timeout",
        }

        if not terminal_loss_reason:
            # Near lock is computed in the goat step transition function
            if bool(info.get("near_lock", False)):
                near_lock_component = self._w("NEAR_LOCK_BONUS") * decay_mult

            if delta_moves > 0:
                block_component = (
                    abs(self._w("REWARD_BLOCK_TIGER"))
                    * float(np.tanh(delta_moves))
                    * decay_mult
                )
            elif delta_moves < 0:
                block_component = (
                    abs(self._w("REWARD_BLOCK_TIGER"))
                    * self._w("MOBILITY_BACKSLIDE_SCALE")
                    * delta_moves
                )

            if delta_bubble > 0:
                bubble_component = (
                    self._w("REWARD_BUBBLE_SPACE")
                    * float(np.tanh(delta_bubble))
                    * decay_mult
                )

            if delta_spread > 0:
                cluster_component = (
                    self._w("REWARD_CLUSTER_TIGERS")
                    * float(np.tanh(delta_spread))
                    * decay_mult
                )

            # Goat center shaping is intentionally limited to the placing phase.
            if phase == 0:
                center_component = float(info.get("center_score", 0.0))

        if goats_eaten_this_turn > 0:
            goat_eaten_component = (
                self._w("REWARD_GOAT_EATEN")
                * goats_eaten_this_turn
            )

        if reason == "tiger_win_capture_threshold":
            terminal_component = self._w("REWARD_TIGER_WIN")
        elif reason == "max_timeout":
            terminal_component = (
                self._w("REWARD_TIGER_WIN")
                * self._w("MAX_TIMEOUT_SCALE")
            )
        elif reason == "repeat_timeout":
            terminal_component = (
                self._w("REWARD_TIGER_WIN")
                * self._w("REPEAT_STALL_SCALE")
            )
        else:
            # Non-terminal repeat discouragement:
            # quadratic in prior repeat count to ramp anti-cycling pressure.
            prev_count = int(info.get("repeat_prev_count", 0))
            if prev_count > 0:
                repeat_component = self._w("REWARD_REPEAT_STATE") * (prev_count ** 2)

        reward_total = (
            step_component
            + near_lock_component
            + block_component
            + bubble_component
            + cluster_component
            + center_component
            + goat_eaten_component
            + repeat_component
            + terminal_component
        )

        self._set_reward_components(
            info,
            reward_step_component=step_component,
            reward_near_lock_component=near_lock_component,
            reward_block_component=block_component,
            reward_bubble_component=bubble_component,
            reward_cluster_component=cluster_component,
            reward_center_component=center_component,
            reward_goat_eaten_component=goat_eaten_component,
            reward_repeat_component=repeat_component,
            reward_terminal_component=terminal_component,
            reward_decay_mult=decay_mult,
            reward_total=reward_total,
        )
        return reward_total


    def _sparse_reward_tiger(self, prev_obs, action, obs, terminated, truncated, info):
        """
        Compute the tiger-learner reward based on the transition summary stored in `info`.

        High-level composition:
        1. Base step penalty (discourages stalling)
        2. Capture rewards (positive reward for eating goats)
        3. Mobility shaping (reward when tiger move options increase)
        4. Positional shaping (center control during placing phase)
        5. Terminal rewards/penalties (win, loss, timeout)

        Important `info` fields consumed here:
        - reason, step_penalty, goats_eaten_this_turn
        - delta_moves, phase, center_score
        """

        # 1. Early exit / invalid action handling
        # These should almost never occur if action masking works correctly.
        # They act as safety guards in case an invalid action slips through.
        reason = info.get("reason")

        if reason == "already_terminated":
            return 0.0

        if reason == "invalid_soft":
            # Small penalty for illegal-but-non-fatal action
            return self._w("REWARD_INVALID_SOFT")

        if reason == "invalid_hard":
            # Illegal tiger actions must be penalized, including when play continues.
            return -abs(self._w("REWARD_INVALID_HARD"))

        # 2. Base step penalty
        # Small negative reward each turn to encourage faster wins
        # and discourage infinite wandering.
        shaped_reward = float(
            info.get("step_penalty", self._w("REWARD_STEP"))
        )

        # 3. Capture reward
        # Tigers receive positive reward when they capture goats.
        goats_eaten_this_turn = int(info.get("goats_eaten_this_turn", 0))

        if goats_eaten_this_turn > 0:
            shaped_reward += (
                self._w("REWARD_TIGER_CAPTURE")
                * goats_eaten_this_turn
            )

        # 4. Tiger mobility shaping
        # delta_moves measures change in number of available tiger moves.
        # Positive delta -> more mobility -> good for tigers.
        delta_moves = int(info.get("delta_moves", 0))

        if delta_moves != 0:
            shaped_reward += (
                abs(self._w("REWARD_BLOCK_TIGER"))
                * delta_moves
            )

        # 5. Center control shaping
        # Reward for occupying strategically strong center nodes. This
        # reward is only active in the placing phase of the game, so the
        # agent doesnt prioritize center control over eating goats.  
        phase = int(info.get("phase", self.phase))
        if phase == 0:
            shaped_reward += float(info.get("center_score", 0.0))

        # 6. Terminal outcome rewards / penalties
        # Large reward when tigers win the game.
        if reason in {"tiger_win_capture_threshold", "tiger_win_no_goat_moves"}:
            shaped_reward += (
                self._w("REWARD_TIGER_WIN_BONUS")
            )

        # Penalty when goats successfully immobilize the tigers.
        elif reason == "goat_win_no_tiger_moves":
            shaped_reward -= (
                self._w("REWARD_TIGER_LOSS_PENALTY")
            )

        # Timeout penalty (scaled loss)
        elif reason == "max_timeout":
            shaped_reward -= (
                self._w("REWARD_TIGER_LOSS_PENALTY")
                * self._w("MAX_TIMEOUT_SCALE")
            )

        return shaped_reward


    #  Group: Transition Engines
    #  Executes full goat-learner and tiger-learner turn transitions.
    #  Applies legality checks, terminal rules, and transition summaries.
    def _step_goat_transition(self, action):
        """
        Run one full goat-learner turn:
          1) Validate and apply goat action
          2) Recompute shaping metrics
          3) Let scripted tiger respond
          4) Finalize terminal checks and build info payload

        Notes:
        - This function returns base reward=0.0; shaping is applied later by
          `_apply_reward_fn(...)` using the `info` summary produced here.
        - `info["reason"]` is the canonical outcome marker used by reward code.
        """
        # Guard against stepping an already-terminal episode.
        if self.terminate:
            return self.get_state(), 0.0, True, False, self._build_step_info(reason="already_terminated")

        # Turn bookkeeping + action normalization for logging/debug.
        self.turns += 1
        flat_action = self._action_to_flat(action)
        decoded = self._safe_decode_action(action)
        base_info = {
            "goat_action": flat_action,
            "goat_decoded": decoded,
        }
        if decoded is None:
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Parse validated action tuple.
        pos, dir_code = decoded
        goats_eaten_before = int(self.eaten)

        goat_desc = ""
        tiger_desc = ""

        if not (0 <= pos < BOARD_SIZE) or not (0 <= dir_code < DIR_CODES):
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Snapshot shaping baselines before applying goat action.
        tiger_moves_before = int(self.prev_tiger_moves)
        bubble_before = int(self.prev_unreachable)
        spread_before = float(self.prev_tiger_spread)

        # Step penalty schedule:
        # - placing phase uses fixed REWARD_STEP
        # - moving phase ramps via MOVE_STEP_* knobs
        if self.phase == 0:
            step_penalty = self._w("REWARD_STEP")
        else:
            step_penalty = max(
                self._w("MOVE_STEP_BASE") + self._w("MOVE_STEP_SLOPE") * self.move_steps,
                self._w("MOVE_STEP_MIN"),
            )

        did_goat_move = False
        if self.phase == 0:
            # Placing-phase legality:
            # dir_code must be 0 and destination must be empty.
            if dir_code != 0 or self.board[pos] != 0:
                if DEBUG_INVALID:
                    print("how did you manage this error?", action)
                info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
                self.terminate = True
                return self.get_state(), 0.0, False, True, info

            self.board[pos] = 1
            self.goats_placed += 1
            goat_desc = f"Goat placed at {idx_to_coord(pos)}"
            if self.goats_placed >= TOTAL_GOATS_TO_PLACE:
                # Transition from placing -> moving once all goats are placed.
                self.phase = 1
        else:
            # Moving-phase legality:
            # source must contain goat, destination must exist and be empty.
            dest = self.move_map.get(pos, {}).get(dir_code, None)
            if dest is None or self.board[pos] != 1 or self.board[dest] != 0:
                if DEBUG_INVALID:
                    print("Invalid goat move:", action)
                self.terminate = True
                info = self._build_step_info(
                    reason="invalid_hard",
                    invalid_action=True,
                    error="Invalid move",
                    **base_info,
                )
                return self.get_state(), 0.0, False, True, info

            self.board[pos] = 0
            self.board[dest] = 1
            did_goat_move = True
            goat_desc = f"Goat {idx_to_coord(pos)} -> {idx_to_coord(dest)}"

        if did_goat_move:
            self.move_steps += 1

        # Recompute tiger options immediately after goat move.
        tiger_options = self._tiger_moves(include_dir=False)
        tiger_moves_after = len(tiger_options)
        self.prev_tiger_moves = tiger_moves_after

        # Derived shaping signals consumed by goat reward:
        # near_lock, late-game factor, mobility delta, bubble delta, spread delta.
        near_lock = (0 < tiger_moves_after <= 2)
        late = min(
            max(self.turns - self._w("LATE_GAME_START_TURN"), 0)
            / (self._w("MAX_TURNS") - self._w("LATE_GAME_START_TURN")),
            1.0,
        )
        delta_moves = tiger_moves_before - tiger_moves_after

        bubble_after = self._compute_unreachable_safe_cells()
        self.prev_unreachable = bubble_after
        delta_bubble = bubble_after - bubble_before

        spread_after = self._tiger_spread()
        self.prev_tiger_spread = spread_after
        delta_spread = spread_before - spread_after

        # Center control score (sign/magnitude baked into knob constants).
        center_score = 0.0
        for idx in KEY_CENTERS:
            if self.board[idx] == 1:
                center_score += self._w("REWARD_CENTER_GOAT")
            elif self.board[idx] == 2:
                center_score += self._w("REWARD_CENTER_TIGER")

        common = dict(
            step_penalty=float(step_penalty),
            near_lock=bool(near_lock),
            late=float(late),
            delta_moves=int(delta_moves),
            delta_bubble=int(delta_bubble),
            delta_spread=float(delta_spread),
            center_score=float(center_score),
            goats_eaten_this_turn=0,
            repeat_prev_count=0,
            move_steps=int(self.move_steps),
        )

        # Immediate goat terminal win: tiger has no legal reply.
        if tiger_moves_after == 0:
            self.terminate = True
            self.last_move_desc = goat_desc or "Goat moved, then tigers had no moves"
            info = self._build_step_info(reason="goat_win_no_tiger_moves", winner="Goat", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Tiger response (scripted or learned model).
        tiger_options_with_dir = self._tiger_moves(include_dir=True)
        chosen = self._select_tiger_move(tiger_options, tiger_options_with_dir=tiger_options_with_dir)
        if chosen is None:
            if self.tiger_ai == TIGER_AI_MODEL:
                raise RuntimeError(
                    "Tiger model opponent returned no valid action. "
                    "Check tiger_model_predict_fn and mask handling."
                )
            raise RuntimeError(
                f"Tiger policy '{self.tiger_ai}' returned None despite "
                f"{len(tiger_options)} legal moves. State={self.get_state().tolist()}"
            )

        t_from, t_to, took_capture = chosen
        if took_capture:
            jumped_goat_pos = self._find_jumped_goat(t_from, t_to)
            if jumped_goat_pos is not None and self.board[jumped_goat_pos] == 1:
                self.board[jumped_goat_pos] = 0
                self.eaten += 1

        # Apply tiger board move.
        self.board[t_to] = 2
        self.board[t_from] = 0

        if t_from is not None and t_to is not None:
            if took_capture:
                tiger_desc = f"Tiger {idx_to_coord(t_from)} -> {idx_to_coord(t_to)} capture"
            else:
                tiger_desc = f"Tiger {idx_to_coord(t_from)} -> {idx_to_coord(t_to)}"

        goats_eaten_this_turn = int(self.eaten - goats_eaten_before)
        common["goats_eaten_this_turn"] = goats_eaten_this_turn
        common["tiger_move"] = (int(t_from), int(t_to), bool(took_capture))

        # Human-readable move summary used by render().
        if goat_desc and tiger_desc:
            self.last_move_desc = f"{goat_desc} | {tiger_desc}"
        elif goat_desc:
            self.last_move_desc = goat_desc
        elif tiger_desc:
            self.last_move_desc = tiger_desc
        else:
            self.last_move_desc = ""

        # Terminal checks after tiger response.
        if self.eaten >= GOATS_EATEN_FOR_TIGER_WIN:
            self.terminate = True
            info = self._build_step_info(reason="tiger_win_capture_threshold", winner="Tiger", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Global episode timeout.
        if self.turns >= self.max_turns:
            self.terminate = True
            info = self._build_step_info(reason="max_timeout", winner="MaxTimeout", **base_info, **common)
            return self.get_state(), 0.0, False, True, info

        # Repeat-state tracking for anti-cycling timeout logic.
        board_hash = (self.board.tobytes(), self.eaten, self.phase)
        prev_count = int(self.state_history.get(board_hash, 0))
        new_count = prev_count + 1
        self.state_history[board_hash] = new_count

        # Stall timeout if state repeats too often.
        if new_count >= self._w("MAX_REPEATS"):
            self.terminate = True
            common["repeat_prev_count"] = prev_count
            info = self._build_step_info(reason="repeat_timeout", winner="RepeatTimeout", **base_info, **common)
            return self.get_state(), 0.0, False, True, info

        # Non-terminal path: refresh goat legal moves and return transition summary.
        common["repeat_prev_count"] = prev_count
        self._update_valid_moves()
        info = self._build_step_info(**base_info, **common)
        return self.get_state(), 0.0, False, False, info

    def _step_tiger_transition(self, action):
        """
        Run one full tiger-learner turn:
          1) Validate and apply tiger action
          2) Compute tiger-facing shaping features
          3) Let goat opponent respond (model or random)
          4) Apply terminal checks and return transition summary

        Notes:
        - Like goat transition, this returns base reward=0.0 and relies on
          `_apply_reward_fn(...)` + `info` for final reward shaping.
        - Goat response policy is selected by `self.goat_opponent_ai`.
        """
        # Guard against stepping an already-terminal episode.
        if self.terminate:
            return self.get_state(), 0.0, True, False, self._build_step_info(reason="already_terminated")

        # Turn bookkeeping + action normalization for logging/debug.
        self.turns += 1
        flat_action = self._action_to_flat(action)
        decoded = self._safe_decode_action(action)
        base_info = {
            "goat_action": flat_action,
            "goat_decoded": decoded,
        }
        if decoded is None:
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Parse validated action tuple.
        pos, dir_code = decoded
        goats_eaten_before = int(self.eaten)

        if not (0 <= pos < BOARD_SIZE) or not (0 <= dir_code < DIR_CODES):
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Build legal tiger action table keyed by (from_pos, dir_code).
        # This ensures action IDs map deterministically to legal transitions.
        legal = {}
        for f, t, cap, d in self._tiger_moves(include_dir=True):
            key = (f, d)
            if key in legal:
                raise ValueError(f"duplicate tiger action key (from_pos,dir_code)={key}")
            legal[key] = (t, cap)

        if (pos, dir_code) not in legal:
            self.terminate = True
            info = self._build_step_info(
                reason="invalid_hard",
                invalid_action=True,
                error="Invalid tiger action",
                **base_info,
            )
            return self.get_state(), 0.0, False, True, info

        # Apply validated tiger move.
        dest, took_capture = legal[(pos, dir_code)]
        step_penalty = self._w("REWARD_STEP")

        self.board[dest] = 2
        self.board[pos] = 0

        if took_capture:
            jumped_goat_pos = self._find_jumped_goat(pos, dest)
            if jumped_goat_pos is not None and self.board[jumped_goat_pos] == 1:
                self.board[jumped_goat_pos] = 0
                self.eaten += 1

        # Tiger-facing shaping features:
        # capture count, mobility delta, and center score.
        goats_eaten_this_turn = int(self.eaten - goats_eaten_before)

        tiger_before = int(self.prev_tiger_moves)
        tiger_after = len(self._tiger_moves())
        self.prev_tiger_moves = tiger_after
        delta_moves = tiger_after - tiger_before

        center_score = 0.0
        for idx in KEY_CENTERS:
            if self.board[idx] == 2:
                center_score += abs(self._w("REWARD_CENTER_GOAT"))
            elif self.board[idx] == 1:
                center_score -= abs(self._w("REWARD_CENTER_GOAT"))

        common = dict(
            step_penalty=float(step_penalty),
            goats_eaten_this_turn=goats_eaten_this_turn,
            delta_moves=int(delta_moves),
            center_score=float(center_score),
            tiger_move=(int(pos), int(dest), bool(took_capture)),
        )

        # Immediate tiger terminal win by capture threshold.
        if self.eaten >= GOATS_EATEN_FOR_TIGER_WIN:
            self.terminate = True
            info = self._build_step_info(reason="tiger_win_capture_threshold", winner="Tiger", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Refresh goat legal moves before opponent response.
        self._update_valid_moves()

        # Goat-opponent helper stack:
        # - goat_mask: legal action mask for current board
        # - goat_random_action: uniform legal sample
        # - goat_model_action: model prediction with legality validation
        def goat_mask():
            """Build a legal-action mask for the scripted/model goat opponent."""
            mask = np.zeros(BOARD_SIZE * DIR_CODES, dtype=bool)
            for mv in self.valid_moves:
                if np.isscalar(mv):
                    flat = int(mv)
                else:
                    flat = self.encode_action(mv[0], mv[1])
                if 0 <= flat < mask.size:
                    mask[flat] = True
            return mask

        def goat_random_action():
            """Sample one legal goat action uniformly from the current mask."""
            mask = goat_mask()
            legal_idxs = np.flatnonzero(mask)
            if legal_idxs.size == 0:
                return None
            flat = int(np.random.choice(legal_idxs))
            return flat // DIR_CODES, flat % DIR_CODES

        def goat_model_action():
            """Query goat model policy and return a validated decoded action."""
            if self._goat_model_predict_fn is None:
                return None
            mask = goat_mask()
            obs = self.get_state()
            try:
                flat = int(self._goat_model_predict_fn(obs, mask))
                if 0 <= flat < mask.size and mask[flat]:
                    return flat // DIR_CODES, flat % DIR_CODES
            except Exception:
                return None
            return None

        # Prefer model goat when configured; fallback to random if unavailable
        # or if model proposes an invalid action.
        goat_choice = None
        # Model goat is optional; invalid/missing model prediction falls back to random goat.
        if self.goat_opponent_ai == GOAT_AI_MODEL:
            goat_choice = goat_model_action()
        if goat_choice is None:
            goat_choice = goat_random_action()

        # No legal goat action -> tiger terminal win.
        if goat_choice is None:
            self.terminate = True
            info = self._build_step_info(reason="tiger_win_no_goat_moves", winner="Tiger", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Apply goat response using phase-appropriate legality.
        g_pos, g_dir = goat_choice
        if self.phase == 0:
            if self.board[g_pos] == 0 and g_dir == 0:
                self.board[g_pos] = 1
                self.goats_placed += 1
                if self.goats_placed >= TOTAL_GOATS_TO_PLACE:
                    self.phase = 1
        else:
            g_dest = self.move_map[g_pos].get(g_dir, None)
            if g_dest is not None and self.board[g_pos] == 1 and self.board[g_dest] == 0:
                self.board[g_pos] = 0
                self.board[g_dest] = 1
                self.move_steps += 1

        # Post-response terminal checks.
        if len(self._tiger_moves()) == 0:
            self.terminate = True
            info = self._build_step_info(reason="goat_win_no_tiger_moves", winner="Goat", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Global episode timeout.
        if self.turns >= self.max_turns:
            self.terminate = True
            info = self._build_step_info(reason="max_timeout", winner="MaxTimeout", **base_info, **common)
            return self.get_state(), 0.0, False, True, info

        # Non-terminal path: refresh masks and return transition summary.
        self._update_valid_moves()
        info = self._build_step_info(**base_info, **common)
        return self.get_state(), 0.0, False, False, info


    def _find_jumped_goat(self, t_from: int, t_to: int):
        """
        Given a tiger capture move t_from -> t_to (which must be a jump),
        find the intermediate neighbor that was jumped over.
        Returns jumped_pos or None if not found.
        """
        # Special hub rule for node 0
        if t_from == 0:
            for dir_code, neigh in self.move_map[0].items():
                if neigh is None:
                    continue
                jump = HUB0_CAPTURE_JUMP.get(dir_code, None)
                if jump == t_to:
                    return neigh
            return None

        # Default rule (same dir_code twice)
        for dir_code, neigh in self.move_map.get(t_from, {}).items():
            if neigh is None:
                continue
            jump = self.move_map.get(neigh, {}).get(dir_code, None)
            if jump == t_to:
                return neigh
        return None
    


    #  Group: Tiger Policy Selection
    #  Selects tiger responses for goat-learner turns.
    #  Contains smart heuristic and greedy capture-biased policies.
    def _select_tiger_move(self, tiger_options, tiger_options_with_dir=None):
        """Select tiger move using configured tiger policy (smart, greedy, or model)."""
        if self.learner_role == GOAT_LEARNER:
            # tiger_options here should be 3-tuples
            if tiger_options and len(tiger_options[0]) != 3:
                raise ValueError("Goat-learner tiger_options must be (from,to,cap) 3-tuples")

        if self.tiger_ai == TIGER_AI_MODEL:
            return self._model_tiger_move(tiger_options_with_dir or [])
        if self.tiger_ai == TIGER_AI_SMART:
            return self._smart_tiger_move(tiger_options)
        return self._greedy_tiger_move(tiger_options)

    def _model_tiger_move(self, tiger_options_with_dir):
        """
        Query tiger model opponent and map predicted flat action to a legal tiger move.

        Expected callback signature:
          tiger_model_predict_fn(obs, mask) -> flat action id
        """
        if self._tiger_model_predict_fn is None:
            return None
        if not tiger_options_with_dir:
            return None

        legal = {}
        mask = np.zeros(BOARD_SIZE * DIR_CODES, dtype=bool)
        for from_pos, to_pos, cap, dir_code in tiger_options_with_dir:
            flat = self.encode_action(from_pos, dir_code)
            legal[flat] = (from_pos, to_pos, cap)
            mask[flat] = True

        try:
            flat = int(self._tiger_model_predict_fn(self.get_state(), mask))
        except Exception:
            return None

        return legal.get(flat, None)

    def _smart_tiger_move(self, tiger_options, debug: bool = False):
        """
        Pick a heuristic smart tiger move from legal options.
          1) Take captures first
          2) Try to occupy anchor positions: 0 and one of {9, 10}
          3) Otherwise roam toward stronger board squares
          4) Fallback to any legal move
        """
        board = self.board
        move_map = self.move_map
        tiger_moves = tiger_options

        POS_VALS = {
            0: 8, 1: 4, 2: 6, 3: 7, 4: 7, 5: 6, 6: 4, 7: 4, 8: 7, 9: 8, 10: 8,
            11: 7, 12: 4, 13: 4, 14: 6, 15: 7, 16: 7, 17: 6, 18: 4, 19: 4,
            20: 5, 21: 5, 22: 4,
        }

        def debug_print(*args):
            """Only print diagnostics when debug mode is on."""
            if debug:
                print("[TIGER DEBUG]", *args)

        def is_high_value(pos):
            """Return True if a square is considered worth roaming toward."""
            return POS_VALS[pos] >= 5

        def occupied_by_tiger(pos):
            """Small helper for tiger occupancy checks."""
            return board[pos] == 2

        def bfs_find_next_step(start_pos, target_positions):
            """
            Return the first step on a shortest path from start_pos to any target.
            Returns (first_step, distance), or (None, None) if unreachable.
            """
            if start_pos in target_positions:
                return (None, 0)

            queue = []
            visited = {start_pos}

            # Seed BFS with immediate legal walkable neighbors
            for _, neighbor in move_map[start_pos].items():
                if neighbor is None:
                    continue

                is_walkable = (board[neighbor] == 0) or (neighbor in target_positions)
                if not is_walkable:
                    continue

                if neighbor in target_positions:
                    return (neighbor, 1)

                queue.append((neighbor, neighbor, 1))  # (current, first_step, distance)
                visited.add(neighbor)

            # Expand outward until a target is found
            while queue:
                curr, first_step, dist = queue.pop(0)

                if curr in target_positions:
                    return (first_step, dist)

                for _, neighbor in move_map[curr].items():
                    if neighbor is None or neighbor in visited:
                        continue

                    is_walkable = (board[neighbor] == 0) or (neighbor in target_positions)
                    if not is_walkable:
                        continue

                    visited.add(neighbor)
                    queue.append((neighbor, first_step, dist + 1))

            return (None, None)

        # MAIN LOGIC
        if not tiger_moves:
            debug_print("No legal tiger moves.")
            return None

        debug_print("Board:", board)
        debug_print("Legal tiger moves:", tiger_moves)

        # --- 1) CAPTURE LOGIC ---
        captures = [move for move in tiger_moves if move[2]]
        has_capture = len(captures) > 0

        if has_capture:
            random.shuffle(captures)  # break ties a little before max()
            best_capture = max(captures, key=lambda move: POS_VALS[move[1]])

            debug_print(
                "Capture phase:",
                "candidate captures =", captures,
                "chosen =", best_capture,
                "dest_pos_val =", POS_VALS[best_capture[1]],
            )
            return best_capture

        # --- 2) ANCHOR LOGIC ---
        # The tiger strategy prefers to control two important board regions:
        #   * Anchor 0 (top of the board)
        #   * One of the central anchors {9, 10}
        #
        # These positions tend to give the tigers better mobility and capture lanes.
        # If either anchor is not currently occupied by a tiger, we attempt to move
        # the closest tiger toward that anchor using BFS pathfinding.

        current_tiger_positions = [i for i in range(23) if occupied_by_tiger(i)]

        anchor_0_occupied = occupied_by_tiger(0)
        anchor_primary_occupied = occupied_by_tiger(9) or occupied_by_tiger(10)

        # Determine which anchors we still need to claim
        need_anchor0 = not anchor_0_occupied
        need_primary = not anchor_primary_occupied

        debug_print(
            "Anchor status:",
            f"anchor_0_occupied={anchor_0_occupied}, "
            f"anchor_primary_occupied={anchor_primary_occupied}, "
            f"need_anchor0={need_anchor0}, need_primary={need_primary}",
        )

        if need_anchor0 or need_primary:

            # Build a list of anchor targets that we want to move toward
            # Each entry is (group_name, list_of_target_positions)
            anchor_groups = []

            if need_anchor0:
                anchor_groups.append(("secondary", [0]))

            if need_primary:
                # Prefer moving to an empty primary anchor
                primary_targets = [p for p in (9, 10) if board[p] == 0]

                # If both are occupied (rare but possible during transitions),
                # we still treat them as targets so BFS can plan toward them.
                if not primary_targets:
                    primary_targets = [9, 10]

                anchor_groups.append(("primary", primary_targets))

            debug_print("Anchor groups:", anchor_groups)

            # Track the best candidate anchor move
            # Structure: (distance, tiger_start_pos, first_step, group_name, targets)
            best = None

            for t_pos in current_tiger_positions:

                # If anchor 0 is already held, we avoid moving that tiger away
                if anchor_0_occupied and t_pos == 0:
                    debug_print("Skipping tiger at 0 (already satisfying anchor 0).")
                    continue

                # Same idea for the primary anchors
                if anchor_primary_occupied and t_pos in (9, 10):
                    debug_print(f"Skipping tiger at {t_pos} (already satisfying primary anchor).")
                    continue

                # Try finding a path from this tiger to each anchor group
                for group_name, targets in anchor_groups:

                    step, dist = bfs_find_next_step(t_pos, targets)

                    # If BFS can't reach the anchor through empty spaces, skip
                    if step is None:
                        continue

                    debug_print(
                        f"Anchor candidate: tiger at {t_pos} -> step {step} "
                        f"(group {group_name}, targets {targets}, dist {dist})"
                    )

                    # Choose the tiger that can reach an anchor in the fewest steps
                    if (best is None) or (dist < best[0]):
                        best = (dist, t_pos, step, group_name, targets)

            # Convert the BFS suggestion into a legal tiger move
            if best is not None:
                dist, from_pos, step, group_name, targets = best
                chosen_move = None
                for move in tiger_moves:
                    if move[0] == from_pos and move[1] == step:
                        chosen_move = move
                        break

                if chosen_move is not None:
                    debug_print(
                        "Anchor choice:",
                        f"group={group_name}, targets={targets}, dist={dist},",
                        f"from={from_pos}, to={step}, move={chosen_move}",
                    )
                    return chosen_move

                debug_print(
                    "Anchor choice failed: BFS suggested step not in tiger_moves.",
                    f"group={group_name}, targets={targets}, dist={dist},",
                    f"from={from_pos}, to={step}",
                )
            else:
                debug_print("Anchor logic: no reachable anchor path found, falling through.")

        # --- 3) ROAMING LOGIC ---
        # If no captures or anchor moves are available, the tiger enters a
        # "roaming" mode. The goal here is simple positional improvement:
        #
        #   * Move toward high-value board squares (POS_VALS >= 5)
        #   * Avoid abandoning important anchor positions
        #
        # This encourages the tiger to drift toward strong control squares
        # instead of wandering randomly.

        high_value_moves = []

        for move in tiger_moves:
            t_from, t_to, _ = move

            # Never abandon anchor 0 once we control it
            # This keeps one tiger guarding the top anchor
            if t_from == 0:
                continue

            # Prevent leaving the only occupied primary anchor (9 or 10)
            # If exactly one tiger is holding the central anchor region,
            # we keep it there to maintain positional control.
            leaving_only_primary_anchor = (
                anchor_primary_occupied
                and (
                    (t_from == 9 and board[9] == 2 and board[10] != 2)
                    or (t_from == 10 and board[10] == 2 and board[9] != 2)
                )
            )

            if leaving_only_primary_anchor:
                continue

            # If the destination square has strong positional value,
            # consider it as a roaming candidate
            if is_high_value(t_to):
                high_value_moves.append(move)

        # Choose randomly among strong roaming options
        # (adds slight unpredictability)
        if high_value_moves:
            chosen = random.choice(high_value_moves)

            debug_print(
                "Roaming phase:",
                "high_value_moves =", high_value_moves,
                "chosen =", chosen,
                "dest_pos_val =", POS_VALS[chosen[1]],
            )

            return chosen

        # --- 4) FALLBACK ---
        fallback = random.choice(tiger_moves)
        debug_print(
            "Fallback phase: no captures, no anchor moves, no high-value roam.",
            "Fallback chosen move =", fallback,
        )
        return fallback

    def _greedy_tiger_move(self, tiger_options):
        """Pick a capture-biased random tiger move from legal options."""
        captures = [m for m in tiger_options if m[2]]
        capture_bias = self._w("BASE_TIGER_CAPTURE_BIAS")

        if captures and random.random() < capture_bias:
            return random.choice(captures)

        if tiger_options:
            return random.choice(tiger_options)

        return None
    


    #  Group: Move Generation and Mask Sources
    #  Enumerates legal moves used by masking, transitions, and opponents.
    #  Serves as the source of truth for move legality.
    def _tiger_moves(self, include_dir: bool = False):
        """
        Compute all legal tiger moves.

        Returns:
          list of (from_pos, to_pos, is_capture[, dir_code])

        Deterministic rule:
          - dir_code is always the actual move_map direction used for both:
              (from -> neighbor) and (neighbor -> jump)
          - No special-case direction remapping for node 0.
        """
        moves = []

        for i in range(BOARD_SIZE):
            if self.board[i] != 2:
                continue

            for dir_code, neigh in self.move_map.get(i, {}).items():
                if neigh is None:
                    continue

                # normal move
                if self.board[neigh] == 0:
                    moves.append((i, neigh, False, dir_code) if include_dir else (i, neigh, False))
                    continue

                # capture candidate: goat in neighbor
                if self.board[neigh] != 1:
                    continue

                # Hub constraint from the original x96_TnG.ipynb rules:
                # row-1 nodes (b1..e1) cannot capture "up" through b0.
                if i in (2, 3, 4, 5) and dir_code == 1:
                    continue

                # jump destination
                if i == 0:
                    jump = HUB0_CAPTURE_JUMP.get(dir_code, None)
                else:
                    jump = self.move_map.get(neigh, {}).get(dir_code, None)

                if jump is None or self.board[jump] != 0:
                    continue

                moves.append((i, jump, True, dir_code) if include_dir else (i, jump, True))

        return moves
    def _update_valid_moves(self):
        """
        Recompute all legal goat actions and store in self.valid_moves.

        - During placing phase: any empty cell with dir_code=0
        - During moving phase: goats can move to empty neighbors
        """
        new_moves = []

        if self.phase == 0:
            # placing phase: any empty cell is valid
            for i in range(BOARD_SIZE):
                if self.board[i] == 0:
                    new_moves.append(self.encode_action(i, 0))
        else:
            # moving phase: move goats along legal directions
            for i in range(BOARD_SIZE):
                if self.board[i] == 1:  # goat
                    for dir_code, dest in self.move_map[i].items():
                        if dest is not None and self.board[dest] == 0:
                            new_moves.append(self.encode_action(i, dir_code))

        self.valid_moves = new_moves


    #  Group: Board Metrics and Graph Utilities
    #  Computes reachability and geometry features for shaping signals.
    #  Includes shortest-path helpers on the static board graph.

    def _compute_unreachable_safe_cells(self) -> int:
        """
        Count how many cells are unreachable by any tiger if tigers
        can only walk through empty cells and onto empty cells.
        (Goats act as walls for this reachability notion.)

        This approximates "bubbles": safe pockets tigers can't reach.
        """
        # 1) Find starting tiger positions
        tiger_starts = [i for i in range(BOARD_SIZE) if self.board[i] == 2]

        if not tiger_starts:
            # no tigers: treat all empty/goat cells as unreachable (safe)
            return int(np.sum((self.board == 0) | (self.board == 1)))
        
        # 2) BFS from all tigers through empty cells only
        visited = self._visited_buffer
        visited[:] = False

        q = deque()
        for t in tiger_starts:
            visited[t] = True
            q.append(t)

        while q:
            cur = q.popleft()
            for _, dest in self.move_map.get(cur, {}).items():
                # tiger can walk into empty neighbor (ignore goats in reachability)
                if self.board[dest] == 0 and not visited[dest]:
                    visited[dest] = True
                    q.append(dest)

        # 3) Count unreachable cells that are useful to goats (empty or goat)
        unreachable = 0
        for i in range(BOARD_SIZE):
            if not visited[i] and self.board[i] in (0, 1):
                unreachable += 1
        
        return unreachable
    

    def _precompute_all_pairs_dist(self) -> np.ndarray:
        """
        Precompute BFS shortest path lengths between all board positions
        in the static graph defined by move_map, ignoring pieces.
        """
        dist = np.full((BOARD_SIZE, BOARD_SIZE), BOARD_SIZE, dtype=np.int8)

        for start in range(BOARD_SIZE):
            visited = np.zeros(BOARD_SIZE, dtype=bool)
            q = deque()
            q.append((start, 0))
            visited[start] = True
            dist[start, start] = 0

            while q:
                cur, d = q.popleft()
                for _, dest in self.move_map.get(cur, {}).items():
                    if not visited[dest]:
                        visited[dest] = True
                        dist[start, dest] = d + 1
                        q.append((dest, d + 1))

        return dist
    

    def _shortest_path_len(self, start: int, goal: int) -> int:
        """
        Shortest path length in the static board graph using precomputed
        all-pairs distances.
        """
        return int(self._dist_matrix[start, goal])
    
    
    # we get a positive reward if tiger spread decreases
    def _tiger_spread(self) -> float:
        """
        Sum of pairwise distances between tigers.
        Smaller => tigers more clustered / forced together.
        """
        tiger_positions = [i for i in range(BOARD_SIZE) if self.board[i] == 2]
        if len(tiger_positions) < 2:
            return 0.0

        total = 0.0
        for idx_a in range(len(tiger_positions)):
            for idx_b in range(idx_a + 1, len(tiger_positions)):
                a = tiger_positions[idx_a]
                b = tiger_positions[idx_b]
                total += self._shortest_path_len(a, b)

        return float(total)
# end class TnGEnv()

In [ ]:
# ENV_SOURCE = "repo" replaces the notebook's environment with the repository's env_tng.py,
# so experiments can use rule changes made there without editing the cell above.
if ENV_SOURCE == "repo":
    if REPO is None:
        raise ValueError('ENV_SOURCE = "repo" needs a repository checkout (REPO_DIR or CLONE_REPO in options group I).')
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    import env_tng
    env_tng = importlib.reload(env_tng)
    for _name in dir(env_tng):
        if _name.isupper() or _name in {"TnGEnv", "idx_to_coord"}:
            globals()[_name] = getattr(env_tng, _name)
    print(f"Environment: {REPO / 'env_tng.py'}")
elif ENV_SOURCE == "notebook":
    print("Environment: defined in this notebook (§4)")
else:
    raise ValueError('ENV_SOURCE must be "notebook" or "repo".')
print(f"{len(DEFAULT_KNOBS)} reward/game settings | {BOARD_SIZE} positions | "
      f"{BOARD_SIZE * DIR_CODES} actions | tiger AIs {sorted(VALID_TIGER_AI)}")

### 4b · Environment demo (`ENV_DEMO`)

The first table lists every reward/game setting with its default and the value in effect after
`REWARD_OVERRIDES`. Below it is a game of random legal goat moves against the `DEMO_TIGER` tiger,
printed as text boards (`T` tiger, `G` goat, `0` empty). Set `DEMO_SHOW_EVERY` to print the board
every N moves.

In [ ]:
%%feature ENV_DEMO
unknown = set(REWARD_OVERRIDES) - set(DEFAULT_KNOBS)
if unknown:
    raise ValueError(f"Unknown REWARD_OVERRIDES keys: {sorted(unknown)}")
knobs = pd.DataFrame([{"setting": key, "default": value, "in effect": REWARD_OVERRIDES.get(key, value),
                       "changed": "✱" if key in REWARD_OVERRIDES and REWARD_OVERRIDES[key] != value else ""}
                      for key, value in DEFAULT_KNOBS.items()])
display(knobs.set_index("setting"))

if DEMO_TIGER not in {"greedy", "smart"}:
    raise ValueError('DEMO_TIGER must be "greedy" or "smart".')
set_random_seed(DEMO_SEED)
rng = random.Random(DEMO_SEED)
demo_env = TnGEnv(tiger_ai=DEMO_TIGER, reward_weights=dict(REWARD_OVERRIDES))
demo_env.reset(seed=DEMO_SEED)
demo_env.render()
total, info, moves = 0.0, {}, 0
done = False
while not done:
    action = rng.choice(np.flatnonzero(demo_env.get_action_mask()).tolist())
    _, reward, terminated, truncated, info = demo_env.step(action)
    total += reward
    moves += 1
    done = terminated or truncated
    if DEMO_SHOW_EVERY and moves % DEMO_SHOW_EVERY == 0 and not done:
        print(f"move {moves}: action {action} (pos {action // DIR_CODES}, dir {action % DIR_CODES}) reward {reward:+.3f}")
        demo_env.render()
demo_env.render()
print(f"Random goats vs {DEMO_TIGER} tiger: winner={info.get('winner')} reason={info.get('reason')} "
      f"goats_eaten={info.get('goats_eaten')} moves={moves} return={total:+.3f}")
demo_env.close()

<a id="helpers"></a>

## 5 · Shared helpers (always run)

These three cells only define functions; nothing trains or plays yet.

* **5a · Training and evaluation.** Validates settings, builds masked environments (with opponent
  mixing and saved-model opponents), logs outcomes and reward components to TensorBoard, runs the
  training suite, and evaluates models. Model tigers (`OPPONENT = "model"` for goat learners) and the
  illegal-move outcome are new in this version.
* **5b · Plain PPO with a configurable loss.** Described below.
* **5c · Model tools.** Finds `.zip` models, reads their metadata without loading them, loads
  MaskablePPO *or* PPO models, and chooses actions with or without the legality mask.

`DummyVecEnv` runs the training environments sequentially. Each phase gets its own logs and saved
settings, and its environments are closed after use.

### 5a · Training and evaluation helpers

In [ ]:
SOURCE_COMMIT = "11ee06e215063e64667b9a6e37c16b5ae93a13d1"  # repository commit the environment cell was copied from
NOTEBOOK_VERSION = "1.2"
ALGORITHMS = {"maskable_ppo", "ppo"}
ILLEGAL_HANDLINGS = {"end_game", "lose_turn"}
ACTIVATIONS = {"relu": torch.nn.ReLU, "tanh": torch.nn.Tanh}
OPPONENTS = {"goat": {"greedy", "smart", "model"}, "tiger": {"random", "model"}}
PHASE_KEYS = {"timesteps", "opponent", "mix_prob", "checkpoints_per_run", "goat_model_path",
              "tiger_model_path", "reward_overrides"}


class WinStatsCallback(BaseCallback):
    """
    Windowed win/timeout/illegal-move stats + optional realized opponent tracking.
    If env includes info["tiger_ai"] at episode end, we will track it.
    """
    def __init__(self, log_every_episodes: int = 200, window_episodes: int = 1_000, verbose: int = 0):
        super().__init__(verbose)
        self.log_every_episodes = log_every_episodes
        self.last_logged_episodes = 0
        self.window_episodes = window_episodes

        # cumulative
        self.episodes = 0
        self.goat_wins = 0
        self.tiger_wins = 0
        self.max_timeouts = 0
        self.repeat_timeouts = 0
        self.illegal_moves = 0
        self.attempted_actions = 0
        self.illegal_actions = 0
        self.illegal_action_window = deque(maxlen=1_000)  # last 1000 steps: 1 = illegal attempt

        # rolling window of winners
        self.window = deque(maxlen=window_episodes)

        # rolling window of realized opponent modes (optional)
        self.window_tiger_ai = deque(maxlen=window_episodes)  # values: greedy/smart/model/None
        self.window_goat_ai = deque(maxlen=window_episodes)   # values: random/model/None
        self.reward_component_names = [
            "reward_step_component",
            "reward_near_lock_component",
            "reward_block_component",
            "reward_bubble_component",
            "reward_cluster_component",
            "reward_center_component",
            "reward_goat_eaten_component",
            "reward_repeat_component",
            "reward_terminal_component",
            "reward_decay_mult",
            "reward_total",
        ]
        self.reward_component_windows = {
            name: deque(maxlen=1_000) for name in self.reward_component_names
        }
        self.episode_reward_window = deque(maxlen=window_episodes)
        self.terminal_penalty_windows = {
            "Goat": deque(maxlen=window_episodes),
            "Tiger": deque(maxlen=window_episodes),
            "MaxTimeout": deque(maxlen=window_episodes),
            "RepeatTimeout": deque(maxlen=window_episodes),
        }

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", None)
        infos = self.locals.get("infos", None)

        if infos is not None:
            for info in infos:
                if info is None:
                    continue
                invalid = int(bool(info.get("invalid_action", False)))
                self.attempted_actions += 1
                self.illegal_actions += invalid
                self.illegal_action_window.append(invalid)
                for name in self.reward_component_names:
                    if name in info:
                        self.reward_component_windows[name].append(float(info.get(name, 0.0)))

        if dones is not None and infos is not None:
            for done, info in zip(dones, infos):
                if not done:
                    continue

                winner = outcome_label(info)

                self.episodes += 1
                if winner == "Goat":
                    self.goat_wins += 1
                elif winner == "Tiger":
                    self.tiger_wins += 1
                elif winner == "MaxTimeout":
                    self.max_timeouts += 1
                elif winner == "RepeatTimeout":
                    self.repeat_timeouts += 1
                elif winner == "IllegalMove":
                    self.illegal_moves += 1

                self.window.append(winner)

                # optional: realized opponent mode
                # TnGEnv in the environment cell supplies the episode's tiger mode in info["tiger_ai"].
                self.window_tiger_ai.append(info.get("tiger_ai", None))
                self.window_goat_ai.append(info.get("goat_opponent_ai", info.get("goat_ai", None)))

                episode_info = info.get("episode", None)
                if isinstance(episode_info, dict) and "r" in episode_info:
                    self.episode_reward_window.append(float(episode_info["r"]))

                if winner in self.terminal_penalty_windows:
                    self.terminal_penalty_windows[winner].append(
                        float(info.get("reward_terminal_component", 0.0))
                    )

        # always keep current episode index in logs
        self.logger.record("1.episode_stats (total episodes)/1.episode_index", float(self.episodes))

        if (self.episodes - self.last_logged_episodes) >= self.log_every_episodes and self.episodes > 0:
            total = float(self.episodes)

            self.last_logged_episodes = self.episodes
            # Order of logs (to control TB display grouping)
            self.logger.record("1.episode_stats (total episodes)/2.goat_win_rate", self.goat_wins / total)
            self.logger.record("1.episode_stats (total episodes)/3.tiger_win_rate", self.tiger_wins / total)
            self.logger.record("1.episode_stats (total episodes)/4.max_timeout_rate", self.max_timeouts / total)
            self.logger.record("1.episode_stats (total episodes)/5.repeat_timeout_rate", self.repeat_timeouts / total)
            self.logger.record("1.episode_stats (total episodes)/6.illegal_move_rate", self.illegal_moves / total)
            self.logger.record("6.action_stats/1.illegal_attempt_rate (cumulative)",
                               self.illegal_actions / max(self.attempted_actions, 1))
            self.logger.record("6.action_stats/2.illegal_attempt_rate (per 1000 steps)",
                               sum(self.illegal_action_window) / max(len(self.illegal_action_window), 1))

            w = len(self.window)
            if w > 0:
                for index, (label, outcome) in enumerate([("goat_win_rate", "Goat"), ("tiger_win_rate", "Tiger"),
                                                          ("max_timeout_rate", "MaxTimeout"),
                                                          ("repeat_timeout_rate", "RepeatTimeout")], 2):
                    self.logger.record(f"2.window_stats (per 1000 eps)/{index}.{label}",
                                       sum(1 for x in self.window if x == outcome) / w)
                self.logger.record("2.window_stats (per 1000 eps)/1.episodes_tracked", float(w))

                # optional realized opponent distribution in the last-N window
                tiger_counts = Counter(str(v).lower() for v in self.window_tiger_ai if v is not None)
                denom = tiger_counts["greedy"] + tiger_counts["smart"]
                if denom > 0:
                    self.logger.record("2.window_stats (per 1000 eps)/6.realized_greedy_tiger_frac", tiger_counts["greedy"] / denom)
                    self.logger.record("2.window_stats (per 1000 eps)/7.realized_smart_tiger_frac", tiger_counts["smart"] / denom)

                goat_counts = Counter(str(v).lower() for v in self.window_goat_ai if v is not None)
                denom = goat_counts["model"] + goat_counts["random"]
                if denom > 0:
                    self.logger.record("2.window_stats (per 1000 eps)/8.realized_random_goat_frac", goat_counts["random"] / denom)
                    self.logger.record("2.window_stats (per 1000 eps)/9.realized_model_goat_frac", goat_counts["model"] / denom)
                self.logger.record("2.window_stats (per 1000 eps)/10.illegal_move_rate",
                                   sum(1 for x in self.window if x == "IllegalMove") / w)

            reward_component_labels = {
                "reward_step_component": "1.step_penalty_mean",
                "reward_near_lock_component": "2.near_lock_mean",
                "reward_block_component": "3.block_mean",
                "reward_bubble_component": "4.bubble_mean",
                "reward_cluster_component": "5.cluster_mean",
                "reward_center_component": "6.center_mean",
                "reward_goat_eaten_component": "7.goat_eaten_mean",
                "reward_repeat_component": "8.repeat_mean",
                "reward_terminal_component": "9.terminal_mean",
                "reward_decay_mult": "10.decay_mult_mean",
                "reward_total": "11.total_reward_mean",
            }
            for name, label in reward_component_labels.items():
                values = self.reward_component_windows[name]
                if values:
                    self.logger.record(
                        f"4.reward_components (per 1000 steps)/{label}",
                        sum(values) / float(len(values)),
                    )

            if self.episode_reward_window:
                self.logger.record(
                    "5.episode_reward (per 1000 eps)/1.episode_reward_mean",
                    sum(self.episode_reward_window) / float(len(self.episode_reward_window)),
                )

            terminal_labels = {
                "Goat": "2.goat_terminal_reward_mean",
                "Tiger": "3.tiger_terminal_penalty_mean",
                "MaxTimeout": "4.max_timeout_terminal_penalty_mean",
                "RepeatTimeout": "5.repeat_timeout_terminal_penalty_mean",
            }
            for outcome, label in terminal_labels.items():
                values = self.terminal_penalty_windows[outcome]
                if values:
                    self.logger.record(
                        f"5.episode_reward (per 1000 eps)/{label}",
                        sum(values) / float(len(values)),
                    )

        return True


def outcome_label(info):
    """Goat, Tiger, MaxTimeout, RepeatTimeout, IllegalMove or Other.

    With "lose_turn" handling an illegal attempt on the last turn ends in MaxTimeout, not IllegalMove.
    """
    if info.get("invalid_action") and info.get("winner") != "MaxTimeout":
        return "IllegalMove"
    return info.get("winner") or "Other"


# ── Validation ──────────────────────────────────────────────────────────────
def positive_int(value, label, minimum=1):
    if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
        raise ValueError(f"{label} must be an integer >= {minimum}.")


def finite_number(value, label, low=None, high=None):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError(f"{label} must be a finite number.")
    if (low is not None and value < low) or (high is not None and value > high):
        raise ValueError(f"{label} must be between {low} and {high}.")


def checked_model_path(value, label):
    if not isinstance(value, (str, Path)) or not str(value).strip():
        raise ValueError(f"{label} requires an existing .zip model path.")
    path = resolve_path(value)
    if path.suffix.lower() != ".zip" or not path.is_file():
        raise ValueError(f"{label}: model file not found: {path}")
    return str(path.resolve())


def validate_common():
    if LEARNER_ROLE not in {"goat", "tiger"}:
        raise ValueError('LEARNER_ROLE must be "goat" or "tiger".')
    if not isinstance(EXPERIMENT_NAME, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", EXPERIMENT_NAME):
        raise ValueError("EXPERIMENT_NAME must start with a letter/number and use only letters, numbers, _, -, or .")
    if not isinstance(OUTPUT_ROOT, str) or not OUTPUT_ROOT.strip():
        raise ValueError("OUTPUT_ROOT must be a nonempty path.")
    if DEVICE not in {"cpu", "cuda"}:
        raise ValueError('DEVICE must be "cpu" or "cuda".')
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise ValueError('CUDA is unavailable. Select a GPU runtime or set DEVICE = "cpu".')
    if ALGORITHM not in ALGORITHMS:
        raise ValueError(f"ALGORITHM must be one of {sorted(ALGORITHMS)}.")


def needs_opponent_model(phase, role):
    """Which saved opponent a phase needs: 'goat_model_path', 'tiger_model_path' or None."""
    mix = phase["mix_prob"]
    if role == "tiger" and ((mix is None and phase["opponent"] == "model") or (mix is not None and mix > 0)):
        return "goat_model_path"
    if role == "goat" and mix is None and phase["opponent"] == "model":
        return "tiger_model_path"
    return None


def validate_phase(phase, role):
    phase.setdefault("tiger_model_path", None)  # absent in v1.0 settings files
    positive_int(phase["timesteps"], "timesteps")
    positive_int(phase["checkpoints_per_run"], "checkpoints_per_run", 0)
    if phase["opponent"] not in OPPONENTS[role]:
        raise ValueError(f"{role} learner opponent must be one of {sorted(OPPONENTS[role])}.")
    mix = phase["mix_prob"]
    if mix is not None:
        finite_number(mix, "mix_prob", 0, 1)
    rewards = phase["reward_overrides"]
    if not isinstance(rewards, dict):
        raise ValueError("reward_overrides must be a dictionary.")
    unknown = set(rewards) - set(DEFAULT_KNOBS)
    if unknown:
        raise ValueError(f"Unknown reward/game settings: {sorted(unknown)}")
    for key, value in rewards.items():
        finite_number(value, key)
        if key in {"MAX_TURNS", "MAX_REPEATS"}:
            positive_int(value, key)
    key = needs_opponent_model(phase, role)
    if key:
        phase[key] = checked_model_path(phase[key], key)
    return phase


def base_phase():
    return dict(timesteps=TIMESTEPS, opponent=OPPONENT, mix_prob=MIX_PROB,
                checkpoints_per_run=CHECKPOINTS_PER_RUN, goat_model_path=GOAT_MODEL_PATH,
                tiger_model_path=TIGER_MODEL_PATH, reward_overrides=copy.deepcopy(REWARD_OVERRIDES))


def merge_phase(base, overrides):
    if isinstance(overrides, dict) and "algorithm" in overrides:
        raise ValueError('"algorithm" is a run-level setting: put it beside "phases", not inside a phase.')
    if not isinstance(overrides, dict) or set(overrides) - PHASE_KEYS:
        raise ValueError(f"Phase overrides must be a dictionary with keys: {sorted(PHASE_KEYS)}")
    result = copy.deepcopy(base)
    if not isinstance(result["reward_overrides"], dict) or not isinstance(overrides.get("reward_overrides", {}), dict):
        raise ValueError("reward_overrides must be a dictionary.")
    result.update({k: v for k, v in overrides.items() if k != "reward_overrides"})
    result["reward_overrides"].update(overrides.get("reward_overrides", {}))
    return result


def apply_algorithm_rewards(phase, algorithm):
    """Plain PPO: an illegal move costs PPO_ILLEGAL_MOVE_REWARD unless set explicitly."""
    if algorithm == "ppo" and PPO_ILLEGAL_MOVE_REWARD is not None:
        for key in ("REWARD_INVALID_SOFT", "REWARD_INVALID_HARD"):
            phase["reward_overrides"].setdefault(key, PPO_ILLEGAL_MOVE_REWARD)
    return phase


def normalized_suite():
    """{run_name: (algorithm, [phase, ...])} from TRAINING_RUNS."""
    training_runs = {"baseline": None} if TRAINING_RUNS is None else TRAINING_RUNS
    if not isinstance(training_runs, dict) or not training_runs:
        raise ValueError("TRAINING_RUNS must be None or a nonempty dictionary.")
    suite = {}
    for run_name, run_config in training_runs.items():
        if not isinstance(run_name, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", run_name):
            raise ValueError("Training run names must use letters, numbers, _, -, or . and start with a letter/number.")
        base = base_phase()
        algorithm = ALGORITHM
        if isinstance(run_config, dict) and "algorithm" in run_config:
            run_config = dict(run_config)
            algorithm = run_config.pop("algorithm")
            if algorithm not in ALGORITHMS:
                raise ValueError(f"{run_name}: algorithm must be one of {sorted(ALGORITHMS)}.")
        if run_config is None:
            phases = [{}]
        elif isinstance(run_config, list):
            phases = run_config
        elif isinstance(run_config, dict):
            if "phases" in run_config:
                base = merge_phase(base, {k: v for k, v in run_config.items() if k != "phases"})
                phases = run_config["phases"]
            else:
                phases = [run_config]
        else:
            raise ValueError(f"Invalid training run: {run_name}")
        if not isinstance(phases, list) or not phases:
            raise ValueError(f"{run_name}: phases must be a nonempty list.")
        suite[run_name] = (algorithm, [validate_phase(apply_algorithm_rewards(merge_phase(base, phase_config), algorithm),
                                                      LEARNER_ROLE) for phase_config in phases])
    return suite


def training_seeds():
    seeds = [SEED] if TRAIN_SEEDS is None else TRAIN_SEEDS
    if not isinstance(seeds, list) or not seeds or len(set(seeds)) != len(seeds):
        raise ValueError("TRAIN_SEEDS must be None or a nonempty list of distinct integer seeds.")
    for seed in seeds:
        positive_int(seed, "training seed", 0)
        if seed + NUM_ENVS > 2**32:
            raise ValueError("Each training seed + NUM_ENVS must be <= 2**32.")
    return seeds


def training_jobs():
    """[(run_name, seed, algorithm, phases)]: every training run once per training seed."""
    return [(name, seed, algorithm, phases)
            for name, (algorithm, phases) in normalized_suite().items() for seed in training_seeds()]


def validate_training():
    validate_common()
    positive_int(LOG_INTERVAL, "LOG_INTERVAL")
    finite_number(PRINT_EVERY_SECONDS, "PRINT_EVERY_SECONDS", 0)
    if PRINT_EVERY_SECONDS == 0:
        raise ValueError("PRINT_EVERY_SECONDS must be positive.")
    for name, value in [("NUM_ENVS", NUM_ENVS), ("N_STEPS", N_STEPS), ("N_EPOCHS", N_EPOCHS)]:
        positive_int(value, name)
    jobs = training_jobs()
    if any(algorithm == "ppo" for _, _, algorithm, _ in jobs):
        make_loss_settings()
        if PPO_ILLEGAL_MOVE_REWARD is not None:
            finite_number(PPO_ILLEGAL_MOVE_REWARD, "PPO_ILLEGAL_MOVE_REWARD")
        if ILLEGAL_MOVE_HANDLING not in ILLEGAL_HANDLINGS:
            raise ValueError(f"ILLEGAL_MOVE_HANDLING must be one of {sorted(ILLEGAL_HANDLINGS)}.")
    if RESUME_MODEL_PATH is not None:
        path = checked_model_path(RESUME_MODEL_PATH, "RESUME_MODEL_PATH")
        if len(jobs) != 1:
            raise ValueError("Resume needs exactly one training run and one training seed.")
        if model_algorithm(path) != jobs[0][2]:
            raise ValueError(f"RESUME_MODEL_PATH is a {model_algorithm(path)} model but the run uses {jobs[0][2]!r}.")
    else:
        if ACTIVATION not in ACTIVATIONS:
            raise ValueError(f"ACTIVATION must be one of {sorted(ACTIVATIONS)}.")
        positive_int(BATCH_SIZE, "BATCH_SIZE", 2)
        if N_STEPS * NUM_ENVS < BATCH_SIZE or (N_STEPS * NUM_ENVS) % BATCH_SIZE:
            raise ValueError("BATCH_SIZE must divide N_STEPS * NUM_ENVS and not exceed it.")
        finite_number(LEARNING_RATE, "LEARNING_RATE", 0)
        if LEARNING_RATE == 0:
            raise ValueError("LEARNING_RATE must be positive.")
        finite_number(GAMMA, "GAMMA", 0, 1)
        finite_number(GAE_LAMBDA, "GAE_LAMBDA", 0, 1)
        finite_number(CLIP_RANGE, "CLIP_RANGE", 0)
        if CLIP_RANGE == 0:
            raise ValueError("CLIP_RANGE must be positive.")
        finite_number(ENT_COEF, "ENT_COEF", 0)
        finite_number(VF_COEF, "VF_COEF", 0)
        finite_number(MAX_GRAD_NORM, "MAX_GRAD_NORM", 0)
        if not isinstance(NET_ARCH, list) or not NET_ARCH:
            raise ValueError("NET_ARCH must be a nonempty list of positive widths.")
        for width in NET_ARCH:
            positive_int(width, "NET_ARCH width")
    return jobs


# ── Environments ────────────────────────────────────────────────────────────
class NotebookMatchup(Monitor):
    """Resample opponents at reset using TnGEnv from the environment definition cell."""
    def __init__(self, env, role, mix):
        super().__init__(env)
        self.role, self.mix = role, mix

    def reset(self, **kwargs):
        if self.mix is not None:
            selected = random.random() < self.mix
            if self.role == "goat":
                self.env.tiger_ai = "smart" if selected else "greedy"
            else:
                self.env.goat_opponent_ai = "model" if selected else "random"
        return super().reset(**kwargs)


def make_notebook_env(phase, role, device, illegal_handling=None):
    """A masked TnGEnv for one phase; saved opponents play greedy legal moves.

    illegal_handling: None (environment default: an illegal move ends the game as a truncation),
    "end_game" (ends it as a real ending, for plain PPO training) or "lose_turn" (costs a turn).
    """
    key = needs_opponent_model(phase, role)
    opponent_path = phase[key] if key else None
    predict = opponent_predict_fn(opponent_path, device) if opponent_path else None

    # Mixing selects the actual opponent at every reset, including the first one.
    env = TnGEnv(learner_role=role, reward_weights=phase["reward_overrides"],
                 tiger_ai=phase["opponent"] if role == "goat" else "smart",
                 goat_opponent_ai=phase["opponent"] if role == "tiger" else "random",
                 goat_model_predict_fn=predict if key == "goat_model_path" else None,
                 tiger_model_predict_fn=predict if key == "tiger_model_path" else None,
                 **({"truncate_on_invalid": False} if illegal_handling == "lose_turn" else {}))
    if opponent_path:
        opponent = load_model(opponent_path, device)
        if opponent.observation_space != env.observation_space or opponent.action_space != env.action_space:
            env.close()
            raise ValueError(f"Opponent model has incompatible observation/action spaces: {opponent_path}")
    wrapped = NotebookMatchup(env, role, phase["mix_prob"])
    if illegal_handling == "end_game":
        wrapped = IllegalMoveEndsGame(wrapped)
    return ActionMasker(wrapped, lambda w: w.unwrapped.get_action_mask())


def play_episode(env, model, seed, deterministic=True, use_mask=True, on_step=None):
    """Play one game with `model` as the learner. Returns (final info, total reward, moves)."""
    set_random_seed(seed)
    obs, _ = env.reset(seed=seed)
    done, total, moves, info = False, 0.0, 0, {}
    while not done:
        mask = env.action_masks()
        if not np.any(mask):
            raise RuntimeError("No legal action available before an episode ended.")
        action = choose_action(model, obs, mask, deterministic=deterministic, use_mask=use_mask)
        prev_obs = obs
        obs, reward, terminated, truncated, info = env.step(action)
        total += float(reward)
        moves += 1
        done = terminated or truncated
        if on_step is not None:
            on_step(prev_obs, mask, action, float(reward), info, done)
    return info, total, moves


def matchup_phase(role, opponent, reward_overrides=None):
    """A single fixed-opponent phase. `opponent` is a name or a saved opponent .zip."""
    phase = dict(base_phase(), opponent=opponent, mix_prob=None,
                 reward_overrides=copy.deepcopy(REWARD_OVERRIDES if reward_overrides is None else reward_overrides))
    if str(opponent).lower().endswith(".zip"):
        phase["opponent"] = "model"
        phase["goat_model_path" if role == "tiger" else "tiger_model_path"] = str(opponent)
    return validate_phase(phase, role)



def selected_matchup(opponent, illegal_handling=None):
    """(path, model, role, masked env) for SELECTED_MODEL against `opponent`, with its saved rewards."""
    path = selected_model_path()
    role = infer_role(path)
    settings, _ = settings_for_model(path)
    rewards = settings.get("effective_knobs", REWARD_OVERRIDES) if settings else REWARD_OVERRIDES
    return path, load_model(path, DEVICE), role, make_notebook_env(matchup_phase(role, opponent, rewards), role, DEVICE,
                                                                   illegal_handling)

# ── Training loop ───────────────────────────────────────────────────────────
class TrainingProgressCallback(BaseCallback):
    """Print progress during rollouts without waiting for completed episodes."""
    def __init__(self, label, total_steps, interval_seconds):
        super().__init__()
        self.label, self.total_steps = label, total_steps
        self.interval_seconds = interval_seconds

    def _on_training_start(self):
        self.start_steps = self.num_timesteps
        self.started = self.last_print = monotonic()
        print(f"{self.label}: starting {self.total_steps:,} steps; "
              f"PPO metrics every {LOG_INTERVAL} rollout(s) "
              f"({self.model.n_steps * self.training_env.num_envs:,} steps/rollout).", flush=True)

    def _print_progress(self, now):
        steps = self.num_timesteps - self.start_steps
        elapsed = now - self.started
        fps = steps / max(elapsed, 1e-9)
        print(f"{self.label}: {steps:,}/{self.total_steps:,} steps "
              f"({min(100.0, 100 * steps / self.total_steps):.1f}%) | "
              f"{fps:.0f} steps/s | {elapsed / 60:.1f} min", flush=True)
        self.last_print = now

    def _on_step(self):
        now = monotonic()
        if now - self.last_print >= self.interval_seconds:
            self._print_progress(now)
        return True

    def _on_training_end(self):
        self._print_progress(monotonic())


def unique_run():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid4().hex[:8]


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def create_run_folder(root, run_name):
    root.mkdir(parents=True, exist_ok=True)
    number = 1
    while True:
        folder = root / (run_name if number == 1 else f"{run_name}_{number}")
        try:
            folder.mkdir()  # Reserve a new folder without overwriting an earlier run.
            return folder
        except FileExistsError:
            number += 1


def build_model(vec, algorithm, seed):
    """A fresh or resumed learner."""
    if RESUME_MODEL_PATH:
        path = checked_model_path(RESUME_MODEL_PATH, "RESUME_MODEL_PATH")
        if algorithm == "ppo":
            model = LossConfigurablePPO.load(path, env=vec, device=DEVICE)
            model.loss_settings = make_loss_settings()  # the loss is the experiment variable; apply group B
            if not isinstance(model.rollout_buffer, MaskableRolloutBuffer):  # a PPO model saved elsewhere
                model.rollout_buffer_class = MaskableRolloutBuffer
                model.rollout_buffer = MaskableRolloutBuffer(model.n_steps, model.observation_space, model.action_space,
                                                             device=model.device, gamma=model.gamma,
                                                             gae_lambda=model.gae_lambda, n_envs=model.n_envs)
        else:
            model = MaskablePPO.load(path, env=vec, device=DEVICE)
        model.set_random_seed(seed)
        print("Resuming with saved PPO hyperparameters and architecture; applying the selected seed.")
        return model
    common = dict(device=DEVICE, seed=seed, verbose=1, n_steps=N_STEPS, batch_size=BATCH_SIZE, n_epochs=N_EPOCHS,
                  learning_rate=LEARNING_RATE, gamma=GAMMA, gae_lambda=GAE_LAMBDA, clip_range=CLIP_RANGE,
                  ent_coef=ENT_COEF, vf_coef=VF_COEF, max_grad_norm=MAX_GRAD_NORM,
                  policy_kwargs={"net_arch": NET_ARCH, "activation_fn": ACTIVATIONS[ACTIVATION]})
    if algorithm == "ppo":
        return LossConfigurablePPO("MlpPolicy", vec, loss_settings=make_loss_settings(), **common)
    return MaskablePPO("MlpPolicy", vec, **common)


def run_training():
    jobs = validate_training()  # Validate every phase before creating run folders.
    suite_started = perf_counter()
    torch.set_num_threads(1)
    results = []
    root = Path(OUTPUT_ROOT).expanduser() / EXPERIMENT_NAME
    for run_name, run_seed, algorithm, phases in jobs:
        loss_settings = make_loss_settings() if algorithm == "ppo" else None
        illegal_handling = ILLEGAL_MOVE_HANDLING if algorithm == "ppo" else None
        run_started = perf_counter()
        set_random_seed(run_seed)
        folder = create_run_folder(root, run_name if TRAIN_SEEDS is None else f"{run_name}_seed_{run_seed}")
        for part in ("meta", "tb", "checkpoints", "models", "eval"):
            (folder / part).mkdir(parents=True, exist_ok=True)
        requested = dict(n_steps=N_STEPS, batch_size=BATCH_SIZE, n_epochs=N_EPOCHS, learning_rate=LEARNING_RATE,
                         gamma=GAMMA, gae_lambda=GAE_LAMBDA, clip_range=CLIP_RANGE, ent_coef=ENT_COEF,
                         vf_coef=VF_COEF, max_grad_norm=MAX_GRAD_NORM, net_arch=NET_ARCH, activation=ACTIVATION)
        run_metadata = dict(source_commit=SOURCE_COMMIT, notebook_version=NOTEBOOK_VERSION, algorithm=algorithm,
                            loss_settings=loss_settings, illegal_move_handling=illegal_handling,
                            learner_role=LEARNER_ROLE, seed=run_seed, training_seed=run_seed, device=DEVICE,
                            num_envs=NUM_ENVS, experiment_name=EXPERIMENT_NAME, run_name=run_name,
                            run_folder_name=folder.name, phases=phases, requested_ppo=requested,
                            resume_model_path=RESUME_MODEL_PATH)
        write_json(folder / "meta" / "run.json", run_metadata)
        model = None
        for index, phase in enumerate(phases, 1):
            label = f"[{run_name} seed {run_seed} phase {index}/{len(phases)}]"
            print(f"{label} {algorithm} | {NUM_ENVS} sequential environments | "
                  f"opponent={phase['opponent']}, mix_prob={phase['mix_prob']}, "
                  f"timesteps={phase['timesteps']:,}", flush=True)
            envs, vec, logger = [], None, None
            try:
                for rank in range(NUM_ENVS):
                    envs.append(make_notebook_env(phase, LEARNER_ROLE, DEVICE, illegal_handling))
                vec = DummyVecEnv([lambda env=env: env for env in envs])
                vec.seed(run_seed)
                if model is None:
                    model = build_model(vec, algorithm, run_seed)
                else:
                    model.set_env(vec)
                logger = configure(str(folder / "tb" / f"phase_{index:02}"), ["stdout", "tensorboard"])
                model.set_logger(logger)
                model.verbose = 1  # Also enable metrics for models saved with verbose=0.
                stats = WinStatsCallback(log_every_episodes=10)
                callbacks = [stats, TrainingProgressCallback(label, phase["timesteps"], PRINT_EVERY_SECONDS)]
                if phase["checkpoints_per_run"]:
                    freq = max(phase["timesteps"] // phase["checkpoints_per_run"] // NUM_ENVS, 1)
                    callbacks.append(CheckpointCallback(save_freq=freq, save_path=str(folder / "checkpoints"),
                                                        name_prefix=f"phase_{index:02}"))
                effective = dict(n_steps=model.n_steps, batch_size=model.batch_size, n_epochs=model.n_epochs,
                                 gamma=model.gamma, gae_lambda=model.gae_lambda,
                                 clip_range=float(model.clip_range(model._current_progress_remaining)),
                                 ent_coef=model.ent_coef, vf_coef=model.vf_coef, max_grad_norm=model.max_grad_norm,
                                 net_arch=model.policy.net_arch, activation=model.policy.activation_fn.__name__,
                                 learning_rate_at_start=float(model.lr_schedule(model._current_progress_remaining)))
                settings = dict(source_commit=SOURCE_COMMIT, notebook_version=NOTEBOOK_VERSION, algorithm=algorithm,
                                loss_settings=getattr(model, "loss_settings", None),
                                illegal_move_handling=illegal_handling, learner_role=LEARNER_ROLE, run_name=run_name,
                                phase=phase, effective_knobs={**DEFAULT_KNOBS, **phase["reward_overrides"]},
                                seed=run_seed, training_seed=run_seed, device=DEVICE, num_envs=NUM_ENVS,
                                effective_ppo=effective)
                settings_path = folder / "meta" / f"phase_{index:02}_settings.json"
                write_json(settings_path, settings)
                before = model.num_timesteps
                model.learn(total_timesteps=phase["timesteps"], callback=CallbackList(callbacks),
                            log_interval=LOG_INTERVAL,
                            reset_num_timesteps=(index == 1 and RESUME_MODEL_PATH is None))
                model_path = folder / "models" / f"phase_{index:02}_final.zip"
                model.save(model_path)
                settings.update(actual_timesteps=model.num_timesteps - before,
                                total_model_timesteps=model.num_timesteps,
                                outcomes=dict(goat_wins=stats.goat_wins, tiger_wins=stats.tiger_wins,
                                              max_timeouts=stats.max_timeouts, repeat_timeouts=stats.repeat_timeouts,
                                              illegal_moves=stats.illegal_moves),
                                action_stats=dict(attempted_actions=stats.attempted_actions,
                                                  illegal_actions=stats.illegal_actions,
                                                  illegal_action_rate=stats.illegal_actions / max(stats.attempted_actions, 1)))
                write_json(settings_path, settings)
                print(f"Saved: {model_path}")
            finally:
                if vec is not None:
                    vec.close()
                else:
                    for env in envs:
                        env.close()
                if logger is not None:
                    logger.close()
        elapsed_seconds = perf_counter() - run_started
        run_metadata.update(elapsed_seconds=elapsed_seconds)
        write_json(folder / "meta" / "run.json", run_metadata)
        print(f"{folder.name}: {elapsed_seconds / 60:.2f} minutes")
        results.append(dict(run_name=run_name, training_seed=run_seed, algorithm=algorithm,
                            run_folder_name=folder.name, model_path=str(model_path.resolve()),
                            settings_path=str(settings_path.resolve()), settings=settings,
                            eval_dir=str((folder / "eval").resolve()), actual_timesteps=model.num_timesteps,
                            elapsed_seconds=elapsed_seconds))
    total_elapsed_seconds = perf_counter() - suite_started
    summary = dict(total_elapsed_seconds=total_elapsed_seconds,
                   training_runs={r["run_folder_name"]: dict(elapsed_seconds=r["elapsed_seconds"], model_path=r["model_path"],
                                                             algorithm=r["algorithm"], training_seed=r["training_seed"])
                                  for r in results})
    # Keep a separate summary for each execution of the full training run dictionary.
    number = 1
    while True:
        suffix = "" if number == 1 else f"_{number}"
        summary_path = root / f"training_summary{suffix}.json"
        try:
            stream = summary_path.open("x", encoding="utf-8")
        except FileExistsError:
            number += 1
            continue
        with stream:
            json.dump(summary, stream, indent=2)
        break
    print(f"Total experiment time: {total_elapsed_seconds / 60:.2f} minutes")
    print(f"Timing summary: {summary_path}")
    return results


# ── Evaluation ──────────────────────────────────────────────────────────────
def target_for_model(path, settings_path=None):
    """An evaluation target: model path + the role/phase/reward settings it was trained with."""
    path = checked_model_path(path, "evaluation model")
    if settings_path is not None:
        settings_path = resolve_path(settings_path)
        if not settings_path.is_file():
            raise ValueError(f"EVAL_SETTINGS_PATH does not exist: {settings_path}")
        settings = json.loads(settings_path.read_text(encoding="utf-8"))
        if not isinstance(settings, dict) or settings.get("learner_role") not in {"goat", "tiger"} or not isinstance(settings.get("phase"), dict):
            raise ValueError("EVAL_SETTINGS_PATH must contain a saved phase settings object.")
    else:
        settings, settings_path = settings_for_model(path)
    if settings is None:
        role = infer_role(path)
        phase = base_phase()
        if role != LEARNER_ROLE:  # the global OPPONENT belongs to the other role
            phase.update(opponent="greedy" if role == "goat" else "random", mix_prob=None)
        settings = dict(learner_role=role, phase=phase, source_commit=None)
    elif settings.get("source_commit") != SOURCE_COMMIT:
        print(f"Note: {Path(path).name} was trained with a different environment source commit.")
    folder = Path(path).parent.parent / "eval" if settings_path else Path(OUTPUT_ROOT).expanduser() / "eval"
    return dict(run_name=Path(path).parent.parent.name if settings_path else Path(path).stem,
                model_path=path, settings=settings, eval_dir=str(folder))


def evaluation_targets(training_results):
    choice = EVAL_MODELS
    if choice == "auto":
        choice = "trained" if training_results else "selected"
    if choice == "trained":
        if not training_results:
            raise ValueError('EVAL_MODELS = "trained" needs models trained in this session (turn on TRAIN).')
        return copy.deepcopy(training_results)
    if choice == "selected":
        paths = [selected_model_path()]
    elif choice == "all":
        paths = model_files()
    elif isinstance(choice, (list, tuple)) and choice:
        paths = [resolve_model(name) for name in choice]
    else:
        raise ValueError('EVAL_MODELS must be "auto", "trained", "selected", "all" or a nonempty list.')
    if EVAL_SETTINGS_PATH is not None and len(paths) != 1:
        raise ValueError("EVAL_SETTINGS_PATH applies to exactly one evaluated model.")
    return [target_for_model(path, EVAL_SETTINGS_PATH) for path in paths]


def eval_mask_modes():
    """EVAL_USE_MASK as a list of modes: [True], [False] or [True, False]."""
    modes = EVAL_USE_MASK if isinstance(EVAL_USE_MASK, list) else [EVAL_USE_MASK]
    if not modes or any(not isinstance(m, bool) for m in modes) or len(set(modes)) != len(modes):
        raise ValueError("EVAL_USE_MASK must be True, False or [True, False].")
    return modes


def eval_illegal_handling():
    """How raw-policy evaluation treats an illegal move (EVAL_ILLEGAL_MOVE)."""
    if EVAL_ILLEGAL_MOVE not in ILLEGAL_HANDLINGS:
        raise ValueError(f"EVAL_ILLEGAL_MOVE must be one of {sorted(ILLEGAL_HANDLINGS)}.")
    return None if EVAL_ILLEGAL_MOVE == "end_game" else "lose_turn"


def run_evaluation(training_results):
    validate_common()
    positive_int(EVAL_EPISODES, "EVAL_EPISODES")
    positive_int(EVAL_SEED, "EVAL_SEED", 0)
    if EVAL_SEED + EVAL_EPISODES > 2**32:
        raise ValueError("EVAL_SEED + EVAL_EPISODES must be <= 2**32.")
    if not isinstance(EVAL_DETERMINISTIC, bool):
        raise ValueError("EVAL_DETERMINISTIC must be True or False.")
    mask_modes = eval_mask_modes()
    if EVAL_OPPONENTS is not None and (not isinstance(EVAL_OPPONENTS, list) or not EVAL_OPPONENTS or any(not isinstance(v, str) for v in EVAL_OPPONENTS)):
        raise ValueError("EVAL_OPPONENTS must be None or a nonempty list of opponent names or .zip paths.")
    targets = evaluation_targets(training_results)
    jobs, skipped = [], []
    for target in targets:
        settings = target["settings"]
        role = settings["learner_role"]
        phase = copy.deepcopy(settings["phase"])
        phase.setdefault("tiger_model_path", None)
        # Use saved game/reward settings, including defaults, when metadata exists.
        if "effective_knobs" in settings:
            phase["reward_overrides"] = copy.deepcopy(settings["effective_knobs"])
        for key, value in (("goat_model_path", GOAT_MODEL_PATH), ("tiger_model_path", TIGER_MODEL_PATH)):
            if value is not None:
                phase[key] = value  # Allows moved/uploaded opponent files.
        for opponent in ([None] if EVAL_OPPONENTS is None else EVAL_OPPONENTS):
            if opponent is None:
                jobs.append((target, role, validate_phase(phase, role)))
            elif opponent.lower().endswith(".zip") or opponent in OPPONENTS[role]:
                jobs.append((target, role, matchup_phase(role, opponent, phase["reward_overrides"]) if opponent.lower().endswith(".zip")
                             else validate_phase(dict(copy.deepcopy(phase), opponent=opponent, mix_prob=None), role)))
            else:
                skipped.append(f"{Path(target['model_path']).name} ({role}) vs {opponent}")
    if skipped:
        print("Skipped (opponent not valid for that role):", "; ".join(skipped))
    if not jobs:
        raise ValueError("No model/opponent pairs to evaluate.")
    rows = []
    run_id = unique_run()
    for (target, role, phase), use_mask in [(job, mode) for job in jobs for mode in mask_modes]:
        model = load_model(target["model_path"], DEVICE)
        env = make_notebook_env(phase, role, DEVICE, None if use_mask else eval_illegal_handling())
        totals, lengths, counts, attempts = [], [], Counter(), [0]

        def count_illegal(prev_obs, mask, action, reward, info, done):
            attempts[0] += not mask[action]

        try:
            if model.observation_space != env.observation_space or model.action_space != env.action_space:
                raise ValueError("Evaluation model has incompatible observation/action spaces.")
            for episode in range(EVAL_EPISODES):
                info, total, moves = play_episode(env, model, EVAL_SEED + episode, deterministic=EVAL_DETERMINISTIC,
                                                  use_mask=use_mask, on_step=count_illegal)
                counts[outcome_label(info)] += 1
                totals.append(total)
                lengths.append(moves)
        finally:
            env.close()
        learner_name, opponent_name = ("Goat", "Tiger") if role == "goat" else ("Tiger", "Goat")
        opponent = phase["opponent"] if phase["mix_prob"] is None else "mixed"
        if opponent == "model":
            opponent = "model:" + Path(phase["tiger_model_path" if role == "goat" else "goat_model_path"]).stem
        row = dict(run_name=target["run_name"], run_folder=target.get("run_folder_name", target["run_name"]),
                   model=Path(target["model_path"]).name, model_path=target["model_path"],
                   algorithm=model_algorithm(target["model_path"]), learner=role, opponent=opponent,
                   mix_prob=phase["mix_prob"], episodes=EVAL_EPISODES,
                   learner_wins=counts[learner_name], opponent_wins=counts[opponent_name],
                   max_timeouts=counts["MaxTimeout"], repeat_timeouts=counts["RepeatTimeout"],
                   illegal_moves=counts["IllegalMove"], illegal_attempts_per_game=attempts[0] / EVAL_EPISODES,
                   other_endings=sum(v for k, v in counts.items() if k not in {"Goat", "Tiger", "MaxTimeout", "RepeatTimeout", "IllegalMove"}),
                   win_rate=counts[learner_name] / EVAL_EPISODES,
                   mean_reward=float(np.mean(totals)), mean_length=float(np.mean(lengths)),
                   seed=EVAL_SEED, deterministic=EVAL_DETERMINISTIC, use_mask=use_mask,
                   illegal_handling=None if use_mask else EVAL_ILLEGAL_MOVE,
                   evaluation_run_id=run_id)
        folder = Path(target["eval_dir"])
        folder.mkdir(parents=True, exist_ok=True)
        stem = folder / f"evaluation_{run_id}_{len(rows):02}"
        settings = dict(phase=phase, effective_knobs={**DEFAULT_KNOBS, **phase["reward_overrides"]},
                        environment_source_commit=SOURCE_COMMIT, model_source_commit=target["settings"].get("source_commit"),
                        device=DEVICE, evaluation_run_id=run_id)
        write_json(stem.with_suffix(".json"), dict(summary=row, settings=settings,
                   episode_rewards=totals, episode_lengths=lengths))
        with stem.with_suffix(".csv").open("w", newline="", encoding="utf-8") as stream:
            csv_row = {**row, "settings_json": json.dumps(settings, default=str)}
            writer = csv.DictWriter(stream, fieldnames=list(csv_row))
            writer.writeheader()
            writer.writerow(csv_row)
        rows.append(row)
        print(f"{row['model']} vs {row['opponent']} ({'masked' if use_mask else 'raw'}): "
              f"win rate {row['win_rate']:.0%} → {stem}.json/.csv")
    if EVAL_LOG_CSV:
        log_path = Path(OUTPUT_ROOT).expanduser() / EVAL_LOG_CSV
        log_path.parent.mkdir(parents=True, exist_ok=True)
        new = not log_path.exists()
        with log_path.open("a", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            if new:
                writer.writeheader()
            writer.writerows(rows)
        print(f"Appended {len(rows)} row(s) to {log_path}")
    return pd.DataFrame(rows)


print("Training and evaluation helpers ready.")

### 5b · Plain PPO with a configurable loss (`ALGORITHM = "ppo"`)

**Maskable PPO** zeroes the probability of illegal actions *after* the network runs. The network itself
never has to learn that those moves are illegal. **Plain PPO** samples from all 115 actions, so legality
has to be learned. This cell defines `LossConfigurablePPO`, a subclass of Stable-Baselines3 `PPO`. It
uses the same rollouts, GAE and optimizer, and replaces only the loss. Option group B selects it.

With $r = \pi_\theta(a|s) / \pi_{\text{old}}(a|s)$, normalized advantage $\hat A$ and $\epsilon$ = `CLIP_RANGE`:

| `PPO_POLICY_LOSS` | Policy loss (minimized) | Notes |
| --- | --- | --- |
| `"clip"` | $-\mathbb{E}\left[\min\big(r\hat A,\ \text{clip}(r, 1-\epsilon, 1+\epsilon)\hat A\big)\right]$ | Standard PPO; matches SB3 exactly |
| `"kl_penalty"` | $-\mathbb{E}[r\hat A] + \beta\,\mathbb{E}[(r-1) - \log r]$ | PPO-Penalty. $\beta$ doubles when KL > 1.5·`PPO_KL_TARGET` and halves when KL < target/1.5 |
| `"spo"` | $-\mathbb{E}\left[r\hat A - \frac{\lvert\hat A\rvert}{2\epsilon}(r-1)^2\right]$ | Simple Policy Optimization: a smooth penalty keeps $r$ near the trust region instead of zeroing the gradient |

| `PPO_VALUE_LOSS` | Value loss |
| --- | --- |
| `"mse"` | $\mathbb{E}[(V - R)^2]$ (standard) |
| `"huber"` | Huber loss with threshold `PPO_HUBER_DELTA`, less sensitive to the ±3.2 terminal rewards |

**Legality loss** (`PPO_LEGALITY_COEF` > 0). Each step's legal-action mask is stored with the rollout.
A supervised term then teaches the policy where the legal moves are:

| `PPO_LEGALITY_LOSS` | Term added to the loss |
| --- | --- |
| `"nll"` | $-\mathbb{E}\left[\log \sum_{a \in \text{legal}(s)} \pi_\theta(a|s)\right]$, the cross-entropy to "some legal move" |
| `"mass"` | $\mathbb{E}\left[\sum_{a \notin \text{legal}(s)} \pi_\theta(a|s)\right]$, the probability of an illegal move |

Total loss: policy + `ENT_COEF`·entropy + `VF_COEF`·value + `PPO_LEGALITY_COEF`·legality.

**What changes in the game for plain PPO.** `ILLEGAL_MOVE_HANDLING` chooses:

| Value | An illegal move… | Notes |
| --- | --- | --- |
| `"end_game"` (default) | ends the game | Training marks it as a real ending (`terminated`), so its penalty is not bootstrapped away |
| `"lose_turn"` | costs a turn; the board and opponent stay put | The environment's `truncate_on_invalid=False`, as in `TnG_PPO_Notebook.ipynb`. The game continues to the turn limit, so the agent sees every phase from the start |

Either way the move earns `PPO_ILLEGAL_MOVE_REWARD` (default −3.2, the same as a loss), unless you set
`REWARD_INVALID_SOFT`/`REWARD_INVALID_HARD` yourself. With `"lose_turn"` that penalty repeats on every
attempt, so about −1 is a better starting point. Maskable PPO training is unchanged.

TensorBoard logs `train/illegal_prob` (mean probability of an illegal action) for every plain-PPO run,
plus `train/legality_loss` and `train/kl_beta` where they apply. The win-rate panel also shows
`illegal_move_rate`.

In [ ]:
POLICY_LOSSES = {"clip", "kl_penalty", "spo"}
VALUE_LOSSES = {"mse", "huber"}
LEGALITY_LOSSES = {"nll", "mass"}


def make_loss_settings():
    """Loss options from group B, validated."""
    settings = dict(policy_loss=PPO_POLICY_LOSS, value_loss=PPO_VALUE_LOSS, huber_delta=PPO_HUBER_DELTA,
                    kl_target=PPO_KL_TARGET, kl_beta_init=PPO_KL_BETA,
                    legality_coef=PPO_LEGALITY_COEF, legality_loss=PPO_LEGALITY_LOSS)
    if settings["policy_loss"] not in POLICY_LOSSES:
        raise ValueError(f"PPO_POLICY_LOSS must be one of {sorted(POLICY_LOSSES)}.")
    if settings["value_loss"] not in VALUE_LOSSES:
        raise ValueError(f"PPO_VALUE_LOSS must be one of {sorted(VALUE_LOSSES)}.")
    if settings["legality_loss"] not in LEGALITY_LOSSES:
        raise ValueError(f"PPO_LEGALITY_LOSS must be one of {sorted(LEGALITY_LOSSES)}.")
    for key in ("huber_delta", "kl_target", "kl_beta_init"):
        finite_number(settings[key], key, 0)
        if settings[key] == 0:
            raise ValueError(f"{key} must be positive.")
    finite_number(settings["legality_coef"], "PPO_LEGALITY_COEF", 0)
    return settings


class _RecordActionMasks:
    """Wraps the training callback; stores each step's legal-action mask in the rollout buffer.

    OnPolicyAlgorithm.collect_rollouts calls on_step() after env.step() and before
    rollout_buffer.add(), so buffer.pos is still the slot of the observation that was acted on.
    """
    def __init__(self, callback, env, buffer):
        self.callback, self.env, self.buffer = callback, env, buffer

    def on_rollout_start(self):
        self.masks = get_action_masks(self.env)  # masks for model._last_obs
        self.callback.on_rollout_start()

    def update_locals(self, locals_):
        self.callback.update_locals(locals_)

    def on_step(self):
        self.buffer.action_masks[self.buffer.pos] = self.masks.reshape(self.buffer.n_envs, -1)
        self.masks = get_action_masks(self.env)  # masks for new_obs (after any auto-reset)
        return self.callback.on_step()

    def on_rollout_end(self):
        self.callback.on_rollout_end()


class LossConfigurablePPO(PPO):
    """Stable-Baselines3 PPO (no action masking) with a selectable policy, value and legality loss."""

    def __init__(self, *args, loss_settings=None, **kwargs):
        self.loss_settings = dict(loss_settings or make_loss_settings())
        self.kl_beta = float(self.loss_settings["kl_beta_init"])
        kwargs.setdefault("rollout_buffer_class", MaskableRolloutBuffer)  # adds an action_masks field
        super().__init__(*args, **kwargs)

    def collect_rollouts(self, env, callback, rollout_buffer, n_rollout_steps):
        if not isinstance(rollout_buffer, MaskableRolloutBuffer):
            raise TypeError("LossConfigurablePPO needs rollout_buffer_class=MaskableRolloutBuffer.")
        return super().collect_rollouts(env, _RecordActionMasks(callback, env, rollout_buffer),
                                        rollout_buffer, n_rollout_steps)

    def _evaluate(self, obs, actions):
        """ActorCriticPolicy.evaluate_actions, also returning the full distribution."""
        policy = self.policy
        features = policy.extract_features(obs)
        if policy.share_features_extractor:
            latent_pi, latent_vf = policy.mlp_extractor(features)
        else:
            pi_features, vf_features = features
            latent_pi = policy.mlp_extractor.forward_actor(pi_features)
            latent_vf = policy.mlp_extractor.forward_critic(vf_features)
        distribution = policy._get_action_dist_from_latent(latent_pi)
        return policy.value_net(latent_vf).flatten(), distribution.log_prob(actions), distribution.entropy(), distribution

    def train(self):
        settings = self.loss_settings
        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)
        clip_range = self.clip_range(self._current_progress_remaining)
        if self.clip_range_vf is not None:
            clip_range_vf = self.clip_range_vf(self._current_progress_remaining)

        logs = {name: [] for name in ("entropy", "pg", "value", "legality", "illegal_prob", "clip_fraction", "kl")}
        continue_training = True
        for epoch in range(self.n_epochs):
            for rollout_data in self.rollout_buffer.get(self.batch_size):
                actions = rollout_data.actions.long().flatten()
                values, log_prob, entropy, distribution = self._evaluate(rollout_data.observations, actions)

                advantages = rollout_data.advantages
                if self.normalize_advantage and len(advantages) > 1:
                    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
                log_ratio = log_prob - rollout_data.old_log_prob
                ratio = torch.exp(log_ratio)
                kl = ((ratio - 1) - log_ratio).mean()  # Schulman's k3 estimator of KL(old || new)

                # Policy loss
                if settings["policy_loss"] == "clip":
                    policy_loss = -torch.min(advantages * ratio,
                                             advantages * torch.clamp(ratio, 1 - clip_range, 1 + clip_range)).mean()
                elif settings["policy_loss"] == "kl_penalty":
                    policy_loss = -(advantages * ratio).mean() + self.kl_beta * kl
                else:  # "spo"
                    policy_loss = -(advantages * ratio - advantages.abs() / (2 * clip_range) * (ratio - 1) ** 2).mean()

                # Value loss
                values_pred = values if self.clip_range_vf is None else rollout_data.old_values + torch.clamp(
                    values - rollout_data.old_values, -clip_range_vf, clip_range_vf)
                if settings["value_loss"] == "huber":
                    value_loss = F.huber_loss(values_pred, rollout_data.returns, delta=settings["huber_delta"])
                else:
                    value_loss = F.mse_loss(rollout_data.returns, values_pred)

                entropy_loss = -torch.mean(entropy)

                # Legality loss: probability the policy puts on legal actions
                legal = rollout_data.action_masks.bool()
                legal = legal | ~legal.any(dim=1, keepdim=True)  # rows without a recorded mask count as all legal
                log_p_legal = torch.logsumexp(distribution.distribution.logits.masked_fill(~legal, -torch.inf), dim=1)
                p_illegal = (1 - log_p_legal.exp()).clamp_min(0)
                legality_loss = -log_p_legal.mean() if settings["legality_loss"] == "nll" else p_illegal.mean()

                loss = (policy_loss + self.ent_coef * entropy_loss + self.vf_coef * value_loss
                        + settings["legality_coef"] * legality_loss)

                with torch.no_grad():
                    approx_kl = kl.item()
                    for name, value in (("entropy", entropy_loss), ("pg", policy_loss), ("value", value_loss),
                                        ("legality", legality_loss), ("illegal_prob", p_illegal.mean()),
                                        ("clip_fraction", (torch.abs(ratio - 1) > clip_range).float().mean())):
                        logs[name].append(value.item())
                    logs["kl"].append(approx_kl)

                if self.target_kl is not None and approx_kl > 1.5 * self.target_kl:
                    continue_training = False
                    if self.verbose >= 1:
                        print(f"Early stopping at step {epoch} due to reaching max kl: {approx_kl:.2f}")
                    break

                self.policy.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
                self.policy.optimizer.step()

            self._n_updates += 1
            if not continue_training:
                break

        mean_kl = float(np.mean(logs["kl"])) if logs["kl"] else 0.0
        if settings["policy_loss"] == "kl_penalty":  # adaptive beta (Schulman et al., 2017, sec. 4)
            if mean_kl > 1.5 * settings["kl_target"]:
                self.kl_beta = min(self.kl_beta * 2, 1e4)
            elif mean_kl < settings["kl_target"] / 1.5:
                self.kl_beta = max(self.kl_beta / 2, 1e-4)
            self.logger.record("train/kl_beta", self.kl_beta)

        self.logger.record("train/entropy_loss", np.mean(logs["entropy"]))
        self.logger.record("train/policy_gradient_loss", np.mean(logs["pg"]))
        self.logger.record("train/value_loss", np.mean(logs["value"]))
        self.logger.record("train/approx_kl", mean_kl)
        self.logger.record("train/clip_fraction", np.mean(logs["clip_fraction"]))
        self.logger.record("train/illegal_prob", np.mean(logs["illegal_prob"]))
        if settings["legality_coef"] > 0:
            self.logger.record("train/legality_loss", np.mean(logs["legality"]))
        self.logger.record("train/loss", loss.item())
        self.logger.record("train/explained_variance",
                           explained_variance(self.rollout_buffer.values.flatten(), self.rollout_buffer.returns.flatten()))
        self.logger.record("train/n_updates", self._n_updates, exclude="tensorboard")
        self.logger.record("train/clip_range", clip_range)
        if self.clip_range_vf is not None:
            self.logger.record("train/clip_range_vf", clip_range_vf)


class IllegalMoveEndsGame(gym.Wrapper):
    """Plain PPO training: an illegal move is a real ending (terminated), not a time-limit truncation.

    TnGEnv reports invalid actions as truncated; SB3 would then bootstrap the value of the
    unchanged board and the illegal move would cost almost nothing.
    """
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        if info.get("invalid_action"):
            terminated, truncated = True, False
        return obs, reward, terminated, truncated, info


print("LossConfigurablePPO ready:", f"policy losses {sorted(POLICY_LOSSES)}, value losses {sorted(VALUE_LOSSES)}, "
      f"legality losses {sorted(LEGALITY_LOSSES)}")

### 5c · Model tools

Finding, reading, loading and querying saved models (MaskablePPO or PPO).

In [ ]:
# ── Finding models ──────────────────────────────────────────────────────────
def model_files(checkpoints=None):
    """Every .zip model under MODEL_DIRS; periodic checkpoints only when asked."""
    checkpoints = SHOW_CHECKPOINTS if checkpoints is None else checkpoints
    found = set()
    for folder in MODEL_DIRS:
        root = resolve_path(folder)
        if root.is_dir():
            found.update(p.resolve() for p in root.rglob("*.zip")
                         if checkpoints or p.parent.name != "checkpoints")
    return sorted(found, key=lambda p: (str(p.parent), p.name))


def read_model_zip(path):
    """(data, system_info) stored inside an SB3 .zip; no model is loaded."""
    with zipfile.ZipFile(path) as archive:
        data = json.loads(archive.read("data").decode("utf-8"))
        names = archive.namelist()
        system_info = archive.read("system_info.txt").decode("utf-8", "replace") if "system_info.txt" in names else ""
    return data, system_info


def model_algorithm(path):
    """'maskable_ppo' or 'ppo', from the saved policy class."""
    policy_class = read_model_zip(path)[0].get("policy_class", {})
    module = policy_class.get("__module__", "") if isinstance(policy_class, dict) else ""
    return "maskable_ppo" if "maskable" in module else "ppo"


def settings_for_model(path):
    """The phase_NN_settings.json saved beside a model trained by this notebook, if any."""
    path = Path(path)
    match = re.match(r"phase_(\d+)", path.name)
    if match and path.parent.name in {"models", "checkpoints"}:
        candidate = path.parent.parent / "meta" / f"phase_{int(match.group(1)):02}_settings.json"
        if candidate.is_file():
            return json.loads(candidate.read_text(encoding="utf-8")), candidate
    return None, None


def infer_role(path):
    """Learner role of a saved model: saved settings, then folder name, then file name."""
    settings, _ = settings_for_model(path)
    if settings and settings.get("learner_role") in {"goat", "tiger"}:
        return settings["learner_role"]
    for part in reversed(Path(path).parts[:-1]):
        if part.lower() in {"goats", "tigers"}:
            return part.lower()[:-1]
    name = Path(path).name.lower()
    hits = [(name.find(role), role) for role in ("goat", "tiger") if role in name]
    return min(hits)[1] if hits else LEARNER_ROLE


def saved_policy_kwargs(data):
    """policy_kwargs from a model zip. A custom activation makes SB3 store them pickled; decode those."""
    kwargs = data.get("policy_kwargs") or {}
    if isinstance(kwargs, dict) and ":serialized:" in kwargs:
        from stable_baselines3.common.save_util import json_to_data
        kwargs = json_to_data(json.dumps({"policy_kwargs": kwargs})).get("policy_kwargs") or {}
    return kwargs if isinstance(kwargs, dict) else {}


def model_info(path):
    """One row of facts about a saved model, read from its zip and settings."""
    data, system_info = read_model_zip(path)
    kwargs = saved_policy_kwargs(data)
    net = kwargs.get("net_arch")
    activation = getattr(kwargs.get("activation_fn"), "__name__", "Tanh")  # SB3's default when unset
    sb3 = re.search(r"Stable-Baselines3: (\S+)", system_info)
    algorithm = model_algorithm(path)
    loss = (data.get("loss_settings") or {}).get("policy_loss", "clip")
    if algorithm == "maskable_ppo":  # TnG_PPO_Notebook saves training_use_masking=False for unmasked runs
        loss += " (masked)" if data.get("training_use_masking", True) else " (MaskablePPO, trained unmasked)"
    return dict(model=Path(path).name, role=infer_role(path), algorithm=algorithm, loss=loss, activation=activation,
                steps=data.get("num_timesteps"), net="x".join(map(str, net)) if net else "?",
                n_steps=data.get("n_steps"), batch=data.get("batch_size"), gamma=data.get("gamma"),
                ent_coef=data.get("ent_coef"), sb3=sb3.group(1) if sb3 else "?", folder=short_path(Path(path).parent))


def short_path(path):
    """A path relative to the working folder when it is inside it."""
    try:
        return str(Path(path).resolve().relative_to(Path.cwd().resolve()))
    except ValueError:
        return str(path)


def resolve_model(name):
    """A path, a file name, a unique fragment of a name, or an index into model_files(True)."""
    if isinstance(name, Path) or str(name).lower().endswith(".zip"):
        path = resolve_path(name)
        if path.is_file():
            return path.resolve()
    files = model_files(True)
    if isinstance(name, int) or str(name).isdigit():
        listed = model_files()
        if int(name) >= len(listed):
            raise IndexError(f"model index {name} is out of range (0..{len(listed) - 1})")
        return listed[int(name)]
    text = str(name).lower().removesuffix(".zip")
    exact = [f for f in files if f.stem.lower() == text]
    if len(exact) == 1:
        return exact[0]
    hits = exact or [f for f in files if text in str(f).lower()]
    if not hits:
        raise FileNotFoundError(f"No model matches {name!r} under MODEL_DIRS {MODEL_DIRS}.")
    if len(hits) > 1:
        raise ValueError(f"{name!r} is ambiguous: " + ", ".join(str(h) for h in hits[:8]))
    return hits[0]


def selected_model_path():
    """SELECTED_MODEL (options group E) as a path."""
    if SELECTED_MODEL is None:
        raise ValueError("Set SELECTED_MODEL in options group E.")
    return resolve_model(SELECTED_MODEL)


# ── Loading models and choosing actions ────────────────────────────────────
_model_cache = {}


def load_model(path, device="cpu"):
    """Load a MaskablePPO or plain PPO model (cached per file version)."""
    path = Path(path).resolve()
    key = (str(path), path.stat().st_mtime, device)
    if key not in _model_cache:
        cls = MaskablePPO if model_algorithm(path) == "maskable_ppo" else PPO
        _model_cache[key] = cls.load(str(path), device=device)
    return _model_cache[key]


def policy_log_probs(model, obs):
    """Raw action log-probabilities from the policy network, before any legality mask."""
    obs_tensor, _ = model.policy.obs_to_tensor(np.asarray(obs))
    with torch.no_grad():
        dist = model.policy.get_distribution(obs_tensor)  # no mask passed: raw network output
        return dist.distribution.logits[0].double().cpu().numpy()  # torch normalizes logits to log-probs


def policy_probs(model, obs):
    """Raw action probabilities from the policy network, before any legality mask."""
    return np.exp(policy_log_probs(model, obs))


def choose_action(model, obs, mask, deterministic=True, use_mask=True):
    """Pick an action. use_mask=True chooses among legal actions only (how MaskablePPO plays).

    Masking works on log-probabilities, like MaskablePPO, so legal actions with tiny
    probabilities still rank correctly instead of underflowing to 0.
    """
    log_probs = policy_log_probs(model, obs)
    if use_mask:
        log_probs = np.where(np.asarray(mask, dtype=bool), log_probs, -np.inf)
    if deterministic:
        return int(np.argmax(log_probs))
    probs = np.exp(log_probs - log_probs.max())
    return int(np.random.choice(probs.size, p=probs / probs.sum()))


def opponent_predict_fn(path, device="cpu"):
    """Callback for TnGEnv model opponents: (obs, mask) -> legal action, greedy."""
    model = load_model(path, device)

    def predict(obs, mask):
        return choose_action(model, obs, mask, deterministic=True, use_mask=True)
    return predict


print("Model tools ready.", f"{len(model_files())} model(s) under {MODEL_DIRS}.")

<a id="models"></a>

## 6 · Models (`MODELS`, `MODEL_SUMMARY`)

**6a** lists every `.zip` under `MODEL_DIRS`: the shipped policies in `stable_models/models/` and
everything trained under `OUTPUT_ROOT`. The facts come from inside each zip, so no model is loaded and
the list is instant. It also shows which model `SELECTED_MODEL` points to. `WATCH_GAME`, `DIAGNOSTICS`,
`MODEL_SUMMARY`, the GUI, and evaluation with `EVAL_MODELS = "selected"` all use that model. You can name
it by file name, a unique fragment such as `"RobustGoat"`, the `#` from the table, or a full path.

**6b** loads the selected model and prints its network: parameters per component and the layer sizes.
This replaces `model_summary.py` for the common case.

The `role` column comes from the model's saved settings, its folder (`Goats/`, `Tigers/`) or its file
name. `algorithm` comes from the saved policy class.

§6 runs before training, so models trained in this session appear after you rerun 6a (and 6b for
their summary).

In [ ]:
%%feature MODELS
files = model_files()
if not files:
    print(f"No models under {MODEL_DIRS}. Train one, upload one, or set CLONE_REPO = True for the shipped models.")
else:
    rows = []
    for index, path in enumerate(files):
        try:
            row = model_info(path)
        except Exception as exc:
            row = dict(model=path.name, folder=str(path.parent), error=f"unreadable: {exc}")
        rows.append({"#": index, **row})
    with pd.option_context("display.max_colwidth", 60, "display.max_rows", 200):
        display(pd.DataFrame(rows).set_index("#"))

try:
    path = selected_model_path()
    print(f"\nSELECTED_MODEL → {path}\n  role {infer_role(path)} | algorithm {model_algorithm(path)} | "
          f"saved settings: {settings_for_model(path)[1] or 'none (uses the options above)'}")
except (FileNotFoundError, ValueError, IndexError) as exc:
    print(f"\nSELECTED_MODEL: {exc}")

In [ ]:
%%feature MODEL_SUMMARY
path = selected_model_path()
policy = load_model(path).policy


def parameters(module):
    return sum(p.numel() for p in module.parameters())


def layer_sizes(module):
    linears = [m for m in module.modules() if isinstance(m, torch.nn.Linear)]
    return " → ".join([str(linears[0].in_features)] + [str(m.out_features) for m in linears]) if linears else "—"


components = {
    "features extractor": policy.features_extractor,
    "policy MLP (actor)": policy.mlp_extractor.policy_net,
    "value MLP (critic)": policy.mlp_extractor.value_net,
    "action head (115 logits)": policy.action_net,
    "value head": policy.value_net,
}
summary = pd.DataFrame([dict(component=name, parameters=parameters(module), layers=layer_sizes(module))
                        for name, module in components.items()])
summary.loc[len(summary)] = dict(component="total", parameters=parameters(policy), layers="")
print(f"{path.name} | {type(policy).__name__} | net_arch {policy.net_arch} | activation {policy.activation_fn.__name__}")
display(summary.set_index("component"))
print(policy)

<a id="tensorboard"></a>

## 7 · Monitor training with TensorBoard (`TENSORBOARD`)

Run this **before training**. It watches `OUTPUT_ROOT`, including every experiment and phase subfolder.
With `TB_INCLUDE_HISTORICAL = True` it also shows the benchmark logs in `stable_models/Logging`, under
the run prefix `historical/`. Select runs in the sidebar; refresh once training has written its first
logs.

Useful panels: `1.episode_stats` and `2.window_stats` for win, timeout and illegal-move rates;
`4.reward_components`; and `train/` for the losses. Plain PPO runs add `train/illegal_prob`,
`train/legality_loss` and `train/kl_beta`.

In Colab the dashboard appears in the output. Locally the cell prints a URL and embeds the dashboard;
open the URL in a browser if the editor shows a blank frame.

In [ ]:
%%feature TENSORBOARD
from tensorboard import program

# Resolve OUTPUT_ROOT to an absolute path so TensorBoard watches the same folder on repeated runs.
# Creating the folder lets this cell run before training has written any logs.
_tb_logdir = str(Path(OUTPUT_ROOT).expanduser().resolve())
Path(_tb_logdir).mkdir(parents=True, exist_ok=True)
_tb_historical = resolve_path("stable_models/Logging")
if TB_INCLUDE_HISTORICAL and _tb_historical.is_dir():
    _tb_args = ["--logdir_spec", f"current:{_tb_logdir},historical:{_tb_historical.resolve()}"]
else:
    _tb_args = ["--logdir", _tb_logdir]
print("TensorBoard logs:", " ".join(_tb_args[1:]))

if in_colab:
    # Colab's TensorBoard magic supplies the proxy needed to display the remote server.
    shell.run_line_magic("load_ext", "tensorboard")
    shell.run_line_magic("tensorboard", " ".join(shlex.quote(a) for a in _tb_args))
else:
    # One server per log selection, kept across reruns of this cell; restarting the kernel clears them.
    if "_tb_servers" not in globals():
        _tb_servers = {}
    _tb_key = tuple(_tb_args)
    _tb_entry = _tb_servers.get(_tb_key)

    # A saved URL does not guarantee that its server still responds.
    _tb_alive = False
    if _tb_entry is not None:
        try:
            with urlopen(_tb_entry["url"], timeout=3) as response:
                _tb_alive = response.status == 200
        except (OSError, ValueError):
            pass

    if not _tb_alive:
        _tb_app = program.TensorBoard()
        # 127.0.0.1 keeps the dashboard on this computer; port 0 picks a free port.
        _tb_app.configure(argv=["tensorboard", *_tb_args, "--host", "127.0.0.1", "--port", "0"])
        _tb_entry = {"app": _tb_app, "url": _tb_app.launch()}
        _tb_servers[_tb_key] = _tb_entry

    print(f"Open this dashboard URL in your browser: {_tb_entry['url']}")
    # Remote kernels require port forwarding or the notebook host's port proxy.
    display(IFrame(src=_tb_entry["url"], width="100%", height=600))

<a id="train"></a>

## 8 · Train (`TRAIN`)

Trains every entry in `TRAINING_RUNS` once per training seed, with `ALGORITHM` or the run's own
`"algorithm"`. Each run/seed pair is an independent model; its phases continue that model. The table lists the final model and settings file of each run.

| Option | Maskable PPO | Plain PPO |
| --- | --- | --- |
| Actions sampled from | legal moves only | all 115 actions |
| Loss | clipped surrogate + value MSE + entropy | `PPO_POLICY_LOSS` + `PPO_VALUE_LOSS` + entropy + legality term |
| Illegal move | impossible | ends the game or costs a turn (`ILLEGAL_MOVE_HANDLING`), with `PPO_ILLEGAL_MOVE_REWARD` |
| Saved as | `MaskablePPO` zip | `PPO` zip (loads with plain `PPO.load`) |

To compare the two on identical settings, set
`TRAINING_RUNS = {"masked": {"algorithm": "maskable_ppo"}, "plain": {"algorithm": "ppo"}}`, and add
`TRAIN_SEEDS = [42, 43, 44]` for several seeds. Then evaluate with `EVAL_USE_MASK = [True, False]` to see
how much each relies on the mask.

PPO completes whole rollouts, so the actual number of timesteps can exceed `TIMESTEPS`.

In [ ]:
%%feature TRAIN
training_results = run_training()
display(pd.DataFrame([{k: r[k] for k in ("run_name", "model_path", "settings_path", "actual_timesteps", "elapsed_seconds")}
                      for r in training_results]))

<a id="evaluate"></a>

## 9 · Evaluate (`EVALUATE`)

Evaluation measures how well models play; it never changes them. Every model in `EVAL_MODELS` plays
`EVAL_EPISODES` games against every opponent in `EVAL_OPPONENTS`. With more than one model or opponent,
a **models × opponents win-rate table** follows the full results.

| Setting | What it does |
| --- | --- |
| `EVAL_MODELS` | `"auto"`: this session's trained models, otherwise the selected model. Also `"trained"`, `"selected"`, `"all"` (every listed model), or a list of names, fragments or paths. |
| `EVAL_SETTINGS_PATH` | For one saved model: its `phase_*_settings.json`, to reuse its role, reward and game settings. Models trained by this notebook find theirs automatically. Others use the options above, with the role taken from the folder or file name. |
| `EVAL_OPPONENTS` | Opponents to test separately: `"greedy"`, `"smart"`, `"model"` (vs goats), `"random"`, `"model"` (vs tigers), or a saved opponent `.zip`. Opponents that do not fit a model's role are skipped and listed. `None` replays the final training matchup, including its mix. |
| `EVAL_EPISODES`, `EVAL_SEED` | Games per model and opponent, and the first seed. Game *i* uses seed `EVAL_SEED + i`, so every model sees the same games. |
| `EVAL_DETERMINISTIC` | `True`: the most likely action. `False`: sample. |
| `EVAL_USE_MASK` | `True`: choose among legal moves only, the fair comparison for Maskable PPO models. `False`: the raw policy, which can make illegal moves (`illegal_moves`, `illegal_attempts_per_game`). `[True, False]`: both, as separate rows. Use raw play to check whether a model has learned legality. |
| `EVAL_ILLEGAL_MOVE` | Raw play only: `"end_game"` (strict; the first illegal move loses) or `"lose_turn"` (costs a turn, matching `TnG_PPO_Notebook.ipynb`). |
| `EVAL_LOG_CSV` | Every result row is appended to this file under `OUTPUT_ROOT`, so results accumulate across sessions. |

For example, 3 models × 2 opponents × 100 games = 600 games. Each comparison also saves JSON/CSV files
with its settings in the model's `eval/` folder. A short test checks that everything runs; use more
games before judging model strength.

In [ ]:
%%feature EVALUATE
evaluation_results = run_evaluation(globals().get("training_results", []))
columns = ["run_folder", "model", "algorithm", "learner", "opponent", "use_mask", "mix_prob", "episodes",
           "learner_wins", "opponent_wins", "max_timeouts", "repeat_timeouts", "illegal_moves",
           "illegal_attempts_per_game", "other_endings", "win_rate", "mean_reward", "mean_length"]
display(evaluation_results[columns])
if len(evaluation_results) > 1:
    print("Win rate, models × opponents (and mask mode):")
    evaluation_results["mode"] = evaluation_results["use_mask"].map({True: "masked", False: "raw"})
    display(evaluation_results.pivot_table(index=["run_folder", "model"], columns=["opponent", "mode"],
                                           values="win_rate", aggfunc="first").round(3))

<a id="watch"></a>

## 10 · Watch and diagnose (`WATCH_GAME`, `DIAGNOSTICS`)

Both cells use `SELECTED_MODEL` (§6) and the reward/game settings saved with it, when there are any.
The model's role decides the valid opponents: `"greedy"`, `"smart"` or `"model"` for goats, and
`"random"` or `"model"` for tigers. A saved opponent `.zip` also works.

**10a · Watch one game.** Prints text boards every `WATCH_SHOW_EVERY` moves (`T` tiger, `G` goat,
`0` empty), with each move's reward and the final result. `WATCH_USE_MASK = False` lets the raw policy
make illegal moves; `EVAL_ILLEGAL_MOVE` decides what an illegal move does.

**10b · Diagnostics.** Plays `DIAG_GAMES` games and reports:

1. **Probability on illegal actions.** The raw softmax mass the network puts on illegal moves, by game
   phase, next to a uniform-policy baseline. Maskable PPO hides this mass at play time, but it shows
   whether the network itself knows the rules. Plain PPO's legality loss is meant to push it toward 0.
2. **Outcomes with and without the mask.** The same seeds played twice. If the raw policy loses
   games to illegal moves, the model depends on the mask.
3. **Reward breakdown.** The mean per-game sum of each reward component, showing which shaping terms
   drive the return. Goat learners report every component. Tiger learners show only `step_reward`,
   because the environment does not break the tiger reward into components.

In [ ]:
%%feature WATCH_GAME
path, model, role, watch_env = selected_matchup(WATCH_OPPONENT, None if WATCH_USE_MASK else eval_illegal_handling())
print(f"{path.name} ({role}, {model_algorithm(path)}) vs {WATCH_OPPONENT} | seed {WATCH_SEED} | "
      f"{'legal moves only' if WATCH_USE_MASK else 'raw policy'}")
watch_env.reset(seed=WATCH_SEED)
watch_env.unwrapped.render()
moves = []


def show_move(prev_obs, mask, action, reward, info, done):
    moves.append(reward)
    if done or (WATCH_SHOW_EVERY and len(moves) % WATCH_SHOW_EVERY == 0):
        print(f"move {len(moves)}: action {action} (pos {action // DIR_CODES}, dir {action % DIR_CODES}) "
              f"{'' if mask[action] else '[ILLEGAL] '}reward {reward:+.3f}")
        watch_env.unwrapped.render()


try:
    info, total, length = play_episode(watch_env, model, WATCH_SEED, deterministic=True,
                                       use_mask=WATCH_USE_MASK, on_step=show_move)
finally:
    watch_env.close()
print(f"result: {outcome_label(info)} | reason={info.get('reason')} goats_eaten={info.get('goats_eaten')} "
      f"moves={length} return={total:+.3f}")

In [ ]:
%%feature DIAGNOSTICS
positive_int(DIAG_GAMES, "DIAG_GAMES")
path, model, role, diag_env = selected_matchup(DIAG_OPPONENT, eval_illegal_handling())  # raw games follow EVAL_ILLEGAL_MOVE
decisions, components, current_game = [], [], [0]


def record(prev_obs, mask, action, reward, info, done):
    probs = policy_probs(model, prev_obs)
    decisions.append(dict(phase="placing" if prev_obs[-1] == 0 else "moving", legal_actions=int(mask.sum()),
                          illegal_prob=float(probs[~mask].sum())))
    components.append(dict(game=current_game[0], step_reward=reward,
                           **{k: float(v) for k, v in info.items() if k.startswith("reward_") and k != "reward_decay_mult"}))


outcomes = {"legal moves only": Counter(), "raw policy": Counter()}
try:
    for game in range(DIAG_GAMES):
        current_game[0] = game
        info, _, _ = play_episode(diag_env, model, DIAG_SEED + game, use_mask=True, on_step=record)
        outcomes["legal moves only"][outcome_label(info)] += 1
        info, _, _ = play_episode(diag_env, model, DIAG_SEED + game, use_mask=False)
        outcomes["raw policy"][outcome_label(info)] += 1
finally:
    diag_env.close()

print(f"{path.name} ({role}, {model_algorithm(path)}) vs {DIAG_OPPONENT}: {DIAG_GAMES} games, "
      f"{len(decisions)} decisions\n")
frame = pd.DataFrame(decisions)
by_phase = frame.groupby("phase").agg(decisions=("illegal_prob", "size"), mean_illegal_prob=("illegal_prob", "mean"),
                                      mean_legal_actions=("legal_actions", "mean"))
by_phase.loc["all"] = [len(frame), frame["illegal_prob"].mean(), frame["legal_actions"].mean()]
by_phase["uniform_policy_baseline"] = [float((1 - frame.loc[frame["phase"] == p, "legal_actions"] / 115).mean())
                                       if p != "all" else float((1 - frame["legal_actions"] / 115).mean())
                                       for p in by_phase.index]
print("1. Probability the network puts on illegal actions")
display(by_phase.round(4))
print("2. Outcomes on the same seeds, with and without the legality mask")
display(pd.DataFrame(outcomes).fillna(0).astype(int))
print("3. Reward components: mean per-game sum (legal moves only)")
per_game = pd.DataFrame(components).groupby("game").sum().mean()
display(per_game[per_game != 0].to_frame("mean per game").round(4))  # tiger learners only fill step_reward

<a id="play"></a>

## 11 · Play (`PLAY_BOARD`, `LAUNCH_GUI`)

### 11a · Clickable board in the notebook (`PLAY_BOARD`)

Choose your side with `PLAY_AS` and your opponent with `PLAY_OPPONENT` (options group H). Model opponents
use `TIGER_MODEL_PATH` when you play goats, or `GOAT_MODEL_PATH` when you play tigers.

- **Goats:** click an empty position to place a goat. After all 15 are placed, click a goat and then a
  yellow destination.
- **Tigers:** click a tiger and then a yellow destination, including capture landings.
- Click your selected piece again to cancel, or another of your pieces to change the selection. Blue
  pieces are goats and red pieces are tigers. **New game** resets the board.
- A game ends on a win, a timeout or the repetition limit; the status line reports the result.

Game rules and rewards use `REWARD_OVERRIDES`; `PLAY_MAX_TURNS` sets the turn limit. Rerun the last
cell of 11a after changing play settings to replace the old board.

**VS Code:** ipywidgets 8.1.9 has a reported blank-widget issue
([vscode-jupyter#17504](https://github.com/microsoft/vscode-jupyter/issues/17504)). Setup installs
8.1.8 when ipywidgets is missing. If a board stays blank, run `%pip install "ipywidgets==8.1.8"`, restart
the kernel and Run all again.

In [ ]:
%%feature PLAY_BOARD
# Colab needs its widget manager for the clickable board.
if in_colab:
    output.enable_custom_widget_manager()

In [ ]:
%%feature PLAY_BOARD
# Adapted from HumanBoardWrapper in the original x96_TnG.ipynb; game rules stay in TnGEnv above.
class HumanBoardWrapper:
    def __init__(self, env):
        self.env = env
        # The notebook uses one environment class for both human roles.
        self.is_tiger_env = env.learner_role == "tiger"
        self.selected_piece = None
        self.done = False
        self.closed = False
        self.info = {}
        self.message = ""
        self.state, _ = self.env.reset()

        self.status = widgets.HTML()
        self.buttons = [self._make_button(i) for i in range(BOARD_SIZE)]
        self.restart_button = widgets.Button(description="New game", icon="refresh")
        self.restart_button.on_click(self.reset_game)

        # Preserve the original x96_TnG.ipynb board spacing and numbered square buttons.
        self.spacers = [widgets.Label("", layout=widgets.Layout(width=width))
                        for width in ("120px", "40px")]
        self.rows = [
            widgets.HBox([self.spacers[0], self.buttons[0]]),
            widgets.HBox(self.buttons[1:7]),
            widgets.HBox(self.buttons[7:13]),
            widgets.HBox(self.buttons[13:19]),
            widgets.HBox([self.spacers[1], *self.buttons[19:23]]),
        ]
        self.board_ui = widgets.VBox([self.status, *self.rows, self.restart_button])
        self.update_board()

    def _make_button(self, idx):
        button = widgets.Button(description=str(idx),
                                layout=widgets.Layout(width="40px", height="40px"))
        button.on_click(lambda _button, position=idx: self.click(position))
        return button

    def legal_destinations(self, piece_idx):
        """Map clickable destinations to legal actions from the selected piece."""
        mask = self.env.get_action_mask()
        if self.is_tiger_env:
            # Use the environment's actual capture landing and direction rules.
            # The original x96_TnG.ipynb _resolve_tiger_action method is not used here.
            return {
                dest: self.env.encode_action(start, direction)
                for start, dest, _capture, direction in self.env._tiger_moves(include_dir=True)
                if start == piece_idx and mask[self.env.encode_action(start, direction)]
            }
        return {
            dest: self.env.encode_action(piece_idx, direction)
            for direction, dest in self.env.move_map.get(piece_idx, {}).items()
            if dest is not None and mask[self.env.encode_action(piece_idx, direction)]
        }

    def click(self, idx):
        """Place a goat, select a piece, or move to a highlighted destination."""
        if self.done or self.closed:
            return
        self.message = ""
        if not self.is_tiger_env and self.env.phase == 0:
            action = self.env.encode_action(idx, 0)
            if self.env.get_action_mask()[action]:
                self._apply_action(action)
            else:
                self.message = "Choose an empty position to place a goat."
        else:
            own_piece = 2 if self.is_tiger_env else 1
            if idx == self.selected_piece:
                self.selected_piece = None
            elif self.env.board[idx] == own_piece:
                self.selected_piece = idx
                if not self.legal_destinations(idx):
                    self.message = "That piece has no legal moves; select another."
            elif self.selected_piece is not None:
                action = self.legal_destinations(self.selected_piece).get(idx)
                if action is None:
                    self.message = "Choose a yellow destination or select another piece."
                else:
                    self._apply_action(action)
            else:
                self.message = "Select one of your pieces first."
        self.update_board()

    def _apply_action(self, action):
        # One environment step includes the human move and the opponent's reply.
        self.state, _reward, terminated, truncated, self.info = self.env.step(action)
        self.done = bool(terminated or truncated)
        self.selected_piece = None

    def update_board(self):
        destinations = self.legal_destinations(self.selected_piece) if self.selected_piece is not None else {}
        for idx, button in enumerate(self.buttons):
            piece = int(self.env.board[idx])
            button.description = str(idx)
            button.tooltip = f"{COORD_LABELS[idx]} (position {idx})"
            button.style.button_color = {0: None, 1: "lightblue", 2: "lightcoral"}[piece]
            if idx in destinations:
                button.style.button_color = "yellow"
            button.disabled = self.done or self.closed

        role = "Tigers" if self.is_tiger_env else "Goats"
        opponent = (f"{self.env.goat_opponent_ai.title()} Goats" if self.is_tiger_env
                    else f"{self.env.tiger_ai.title()} Tigers")
        phase = "Place goats" if self.env.phase == 0 else "Move goats"
        status = (
            f"<b>You: {role} | Opponent: {escape(opponent)}</b><br>"
            f"{phase} | Goats placed: {self.env.goats_placed}/{TOTAL_GOATS_TO_PLACE} | "
            f"Goats eaten: {self.env.eaten}/{GOATS_EATEN_FOR_TIGER_WIN} | "
            f"Turn: {self.env.turns}/{self.env.max_turns}"
        )
        if self.done:
            winner = self.info.get("winner")
            result = f"{winner} wins" if winner in {"Goat", "Tiger"} else "Game ended"
            reason = self.info.get("reason") or winner or "episode ended"
            status += f"<br><b>{escape(result)}: {escape(str(reason))}. Click New game to play again.</b>"
        elif self.message:
            status += "<br>" + escape(self.message)
        elif not self.is_tiger_env and self.env.phase == 0:
            status += "<br>Click an empty position to place a goat."
        else:
            status += "<br>Select your piece, then click a yellow destination."
        self.status.value = status + (
            '<br><b>Legend:</b> '
            '<span style="background-color: lightblue; color: black; padding: 2px 6px;">Goat</span> | '
            '<span style="background-color: lightcoral; color: black; padding: 2px 6px;">Tiger</span> | '
            '<span style="background-color: yellow; color: black; padding: 2px 6px;">Legal move</span> | '
            'Uncolored = empty'
        )

    def reset_game(self, _button=None):
        if self.closed:
            return
        self.state, _ = self.env.reset()
        self.done = False
        self.selected_piece = None
        self.info = {}
        self.message = ""
        self.update_board()

    def show(self):
        display(self.board_ui)

    def close(self):
        # Rerunning the launch cell retires the old game and its widget callbacks.
        self.closed = True
        self.env.close()
        for widget in [self.board_ui, self.status, self.restart_button, *self.rows, *self.spacers, *self.buttons]:
            widget.close()

In [ ]:
%%feature PLAY_BOARD
if PLAY_AS not in OPPONENTS:
    raise ValueError('PLAY_AS must be "goat" or "tiger".')
if PLAY_OPPONENT not in OPPONENTS[PLAY_AS]:
    raise ValueError(f"Playing as {PLAY_AS}, PLAY_OPPONENT must be one of {sorted(OPPONENTS[PLAY_AS])}.")
positive_int(PLAY_MAX_TURNS, "PLAY_MAX_TURNS")

play_predict = None
if PLAY_OPPONENT == "model":
    key = "TIGER_MODEL_PATH" if PLAY_AS == "goat" else "GOAT_MODEL_PATH"
    play_predict = opponent_predict_fn(checked_model_path(globals()[key], key), DEVICE)

play_env = TnGEnv(learner_role=PLAY_AS, reward_weights=dict(REWARD_OVERRIDES), max_turns=PLAY_MAX_TURNS,
                  tiger_ai=PLAY_OPPONENT if PLAY_AS == "goat" else "smart",
                  goat_opponent_ai=PLAY_OPPONENT if PLAY_AS == "tiger" else "random",
                  tiger_model_predict_fn=play_predict if PLAY_AS == "goat" else None,
                  goat_model_predict_fn=play_predict if PLAY_AS == "tiger" else None)
previous_game = globals().get("human_game")
if previous_game is not None:
    previous_game.close()
human_game = HumanBoardWrapper(play_env)
human_game.show()

### 11b · Tkinter GUI (`LAUNCH_GUI`)

Opens the repository's `gui.py` in its own window. It needs a repository checkout, a display, and
`tkinter` (on Linux, the `python3-tk` package). It does not work in Colab or over SSH.

The GUI offers play as goat, play as tiger, player vs player, replays, model-assisted moves, and
recording and saving replays. `GUI_TIGER` sets the scripted tiger. With `GUI_USE_SELECTED_MODEL`, the
selected model is preloaded when it is a Maskable PPO goat model; you can load others from inside the
GUI. The GUI always uses the repository's `env_tng.py`.

In [ ]:
%%feature LAUNCH_GUI
if GUI_TIGER not in {"greedy", "smart"}:
    raise ValueError('GUI_TIGER must be "greedy" or "smart".')
if REPO is None or not (REPO / "gui.py").is_file():
    print("gui.py needs a repository checkout (REPO_DIR or CLONE_REPO in options group I).")
elif in_colab or (sys.platform.startswith("linux") and not os.environ.get("DISPLAY")):
    print("No display is available here (Colab / SSH), so the GUI cannot open. Use the board in 11a.")
elif importlib.util.find_spec("tkinter") is None:
    print("tkinter is not available in this Python (Linux: install the python3-tk package).")
else:
    gui_command = [sys.executable, str(REPO / "gui.py"), "--env", GUI_TIGER]
    if GUI_USE_SELECTED_MODEL:
        gui_model = selected_model_path()
        if infer_role(gui_model) == "goat" and model_algorithm(gui_model) == "maskable_ppo":
            gui_command += ["--model", str(gui_model)]
        else:
            print(f"Not preloading {gui_model.name}: the GUI preloads Maskable PPO goat models only.")
    GUI_PROC = subprocess.Popen(gui_command, cwd=str(REPO))
    print(f"GUI started (pid {GUI_PROC.pid}); close its window to finish, or run GUI_PROC.terminate().")

<a id="outputs"></a>

## 12 · Outputs and next steps

**Where results go.** Each run lives in `OUTPUT_ROOT/EXPERIMENT_NAME/<training run>/`, with `meta`,
`tb`, `checkpoints`, `models` and `eval` subfolders. Repeated names get a number (`baseline`,
`baseline_2`, …), so earlier runs are kept. Keep each model **and** its `meta/phase_*_settings.json`.
The settings file records the algorithm, the loss settings, every reward/game value in effect, and the
PPO hyperparameters. Evaluation and §6 read it automatically. Every evaluation is also appended to
`OUTPUT_ROOT/evaluation_log.csv`.

**Timing.** Each model's time (all its phases, setup and saving) is `elapsed_seconds` in its
`meta/run.json`. The experiment folder's `training_summary.json` records every model's time and the
session total; later sessions write `_2`, `_3` and so on. Interrupted work has no final time.

**Colab.** Download results from the Files panel before the session ends, or set `OUTPUT_ROOT` to a
mounted Drive folder.

**Resume.** Set `RESUME_MODEL_PATH` (options group D) and the extra timesteps, then restart and Run all.
The saved architecture and PPO hyperparameters are kept. This is continued training, not a bit-for-bit
replay of the original random state.

**Snapshot.** The environment in §4 is a copy of the repository's `env_tng.py` at commit `11ee06e`
(v1.1: tiger illegal-move fix, `truncate_on_invalid`). To use later repository changes without editing
§4, set `ENV_SOURCE = "repo"`.

**Suggested experiment: does plain PPO learn legality?**
1. Set `TRAINING_RUNS = {"masked": {"algorithm": "maskable_ppo"}, "plain": {"algorithm": "ppo"}}`,
   `PPO_LEGALITY_COEF = 0.5` and `TRAIN_SEEDS = [42, 43]`.
2. Repeat under another `EXPERIMENT_NAME` with `PPO_LEGALITY_COEF = 0`, with
   `PPO_POLICY_LOSS = "spo"` or `"kl_penalty"`, or with `ILLEGAL_MOVE_HANDLING = "lose_turn"`.
3. Evaluate with `EVAL_MODELS = "all"` and `EVAL_USE_MASK = [True, False]`. Compare `illegal_moves`,
   `illegal_attempts_per_game` and `win_rate`. Use §10b for the illegal-action probability by phase.

[Future Plans](Important_Docs/FUTURE_PLANS.ipynb) describes opponent pools, sticky opponent sampling and
randomized starting states. These are not implemented here.